# 4. Liver experiments

Most cells run on CPU. Cells marked GPU need a GPU runtime. All result tables are saved in `MyDrive/CARE2026_ckpts/paper_results`.

**Needs** the outputs of notebooks 1 to 3.

**Analysis plan** (Table 1 of the paper)
- Primary endpoint: concordance (C index) between the risk score P(S2+) + P(S4) and the four level stage, counting only pairs from the same held out scanner, pooled over the three leave one scanner out folds.
- Family 1 (CELL V, Holm correction): fusion, radiomic forest and frozen DINOv2 against the deep stream.
- Family 2 (CELL S2, Holm correction): three interventions against the deep stream, seed matched: GED4 + T2 + DWI_800 input (`multiphase`), scanner style appearance augmentation (`intensity_aug`), and a gradient reversal scanner classifier on the embedding (`adversarial`).
- ComBat (CELL U) as the standard harmonization baseline for the radiomic features.

Some cell titles mention a first or second notebook because we ran several Colab sessions in parallel. Here the cells are in one notebook, in an order that works.

**Step 1.** CELL B: leave one scanner out results for the radiomic forest and fusion, per scanner and task (Table S1).

In [ ]:
# ===================== CELL 1: SETUP + B (LOCO main results) =====================
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, os, warnings
from scipy.stats import rankdata
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, accuracy_score, brier_score_loss
warnings.filterwarnings("ignore")

CKPT = "/content/drive/MyDrive/CARE2026_ckpts"
RES  = f"{CKPT}/paper_results"; os.makedirs(RES, exist_ok=True)
CENTERS = ["A", "B1", "B2"]
TASKS = {"sub": 2, "cir": 4}
PHASES = ['T1','T2','DWI_800','GED1','GED2','GED3','GED4']
FO = ['mean','std','min','max','median','skew','kurt','entropy','iqr','energy']
TX = ['contrast_m','contrast_s','homog_m','homog_s','energy_m','energy_s','corr_m','corr_s','dissim_m','dissim_s']
DR = ['grad_mean','grad_std','sobel_mean','aniso','ax_z','ax_x']
ENR = ['mc_surface','convex_vol','surf_to_hull','compactness2','roi_skew2','roi_kurt2','roi_p10','roi_p90','roi_iqr2']

FEATS = {k: v for k, v in {"ged4":   f"{CKPT}/lifs_stad_ged4_clean.csv",
                           "fixed7": f"{CKPT}/lifs_stad_fixed.csv"}.items() if os.path.exists(v)}
print("feature sets available:", list(FEATS))

dp = pd.read_csv(f"{CKPT}/lifs_loco_deep/loco_deep_preds.csv")
dp["p_sub"] = 1 - dp["d1"]; dp["p_cir"] = dp["d4"]
DEEP = dp.groupby(["filename", "held", "stage"], as_index=False)[["d1","d2","d3","d4","p_sub","p_cir"]].mean()

def label(stage, task): return (np.asarray(stage) >= TASKS[task]).astype(int)
rng = np.random.default_rng(0)
def boot(y, n=2000):
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    return [np.r_[rng.choice(pos, len(pos)), rng.choice(neg, len(neg))] for _ in range(n)]
def auc(y, p, i=None):
    return roc_auc_score(y, p) if i is None else roc_auc_score(y[i], p[i])
def ci(v): return np.percentile(v, [2.5, 97.5])
def load_feats(path):
    df = pd.read_csv(path)
    cols = [c for c in df.columns if c not in ("filename", "vendor", "stage") and df[c].dtype != "object"]
    return df, cols
def center_z(X, V):
    Z = np.full_like(X, np.nan, dtype=float)
    for v in np.unique(V):
        m = V == v; mu = np.nanmean(X[m], 0); sd = np.nanstd(X[m], 0)
        sd[~np.isfinite(sd) | (sd < 1e-8)] = 1.0
        Z[m] = (X[m] - mu) / sd
    return Z
def center_rank(X, V):
    R = np.full_like(X, np.nan, dtype=float)
    for v in np.unique(V):
        idx = np.where(V == v)[0]
        for j in range(X.shape[1]):
            col = X[idx, j]; ok = np.isfinite(col)
            if ok.sum() > 1:
                r = np.full(len(col), np.nan); r[ok] = (rankdata(col[ok]) - 0.5) / ok.sum(); R[idx, j] = r
    return R
VARIANTS = {"raw": lambda X, V: X, "center-z": center_z, "center-rank": center_rank}
def rf(n=600):
    return make_pipeline(SimpleImputer(strategy="median"), FunctionTransformer(np.nan_to_num),
                         RandomForestClassifier(n_estimators=n, min_samples_leaf=2, max_features="sqrt",
                                                class_weight="balanced", n_jobs=-1, random_state=42))
def score(m, X):
    try: return m.predict_proba(X)[:, 1]
    except AttributeError: return m.decision_function(X)

def loco_main(fs, path):
    df, cols = load_feats(path)
    X0, st, V = df[cols].values.astype(float), df["stage"].values, df["vendor"].astype(str).values
    rows, cases = [], []
    for vname, vfn in VARIANTS.items():
        X = vfn(X0, V)
        for held in CENTERS:
            tr = V != held
            te = df.loc[~tr, ["filename", "stage"]].copy()
            for task in TASKS:
                te[f"rf_{task}"] = score(rf().fit(X[tr], label(st[tr], task)), X[~tr])
            m = DEEP[DEEP.held == held].merge(te, on=["filename", "stage"], how="inner")
            if len(m) != (~tr).sum(): print(f"  WARNING {fs}/{held}: matched {len(m)} of {(~tr).sum()}")
            for task in TASKS:
                y = label(m.stage, task); I = boot(y)
                d, r = m[f"p_{task}"].values, m[f"rf_{task}"].values; fu = 0.5*d + 0.5*r
                rec = dict(feat=fs, variant=vname, held=held, task=task, n=len(y), n_neg=int((y == 0).sum()))
                for name, p in [("deep", d), ("rf", r), ("fusion", fu)]:
                    lo, hi = ci([auc(y, p, i) for i in I])
                    rec[name], rec[f"{name}_lo"], rec[f"{name}_hi"] = auc(y, p), lo, hi
                for name, p in [("fus_minus_deep", d), ("fus_minus_rf", r)]:
                    lo, hi = ci([auc(y, fu, i) - auc(y, p, i) for i in I])
                    rec[name], rec[f"{name}_lo"], rec[f"{name}_hi"] = auc(y, fu) - auc(y, p), lo, hi
                rows.append(rec)
            cases.append(m.assign(feat=fs, variant=vname))
    return pd.DataFrame(rows), pd.concat(cases)

S, C = zip(*[loco_main(fs, p) for fs, p in FEATS.items()])
S, C = pd.concat(S), pd.concat(C)
S["fus_minus_best"]  = S["fusion"] - S[["deep", "rf"]].max(1)
S["fus_minus_worst"] = S["fusion"] - S[["deep", "rf"]].min(1)
S.to_csv(f"{RES}/B_loco_summary.csv", index=False); C.to_csv(f"{RES}/B_loco_cases.csv", index=False)
for (fs, vn), g in S.groupby(["feat", "variant"], sort=False):
    print(f"\n=== {fs} | {vn} ===")
    print(g[["held","task","n_neg","deep","rf","fusion","fus_minus_best","fus_minus_worst"]].round(3).to_string(index=False))
    for _, r in g.iterrows():
        print(f"   {r.held:2s} {r.task}: rf [{r.rf_lo:.2f}-{r.rf_hi:.2f}]  "
              f"fus-deep {r.fus_minus_deep:+.3f} [{r.fus_minus_deep_lo:+.3f},{r.fus_minus_deep_hi:+.3f}]")
seed = pd.DataFrame([dict(held=h, task=t, seed=s, auc=auc(label(g.stage, t), g[f"p_{t}"].values))
                     for h in CENTERS for s in sorted(dp.seed.unique())
                     for g in [dp[(dp.held == h) & (dp.seed == s)]] for t in TASKS])
sv = seed.groupby(["held", "task"]).auc.agg(["mean", "std"]).round(3)
print("\n=== deep single-seed AUC, mean +/- std over 3 seeds ===\n", sv)
sv.to_csv(f"{RES}/B_deep_seed_variance.csv")

**Step 2.** CELL M: stable feature selection (not reported in the paper).

In [ ]:
# ===================== CELL 2: M (stable feature selection, method candidate) =====================
from scipy.stats import spearmanr
PRIMARY_TAU, TAUS = 0.1, [0.0, 0.1, 0.15, 0.2]     # declared before running; 0.1 is primary

def stable_cols(X, st, V, train_centers, tau):
    keep = []
    for j in range(X.shape[1]):
        rs = []
        for c in train_centers:
            m = V == c; x = X[m, j]; ok = np.isfinite(x)
            if ok.sum() < 10 or np.std(x[ok]) < 1e-12: rs = None; break
            r = spearmanr(x[ok], st[m][ok])[0]
            if not np.isfinite(r): rs = None; break
            rs.append(r)
        if rs and np.all(np.sign(rs) == np.sign(rs[0])) and min(abs(v) for v in rs) >= tau:
            keep.append(j)
    return keep

lr = lambda: make_pipeline(SimpleImputer(strategy="median"), FunctionTransformer(np.nan_to_num), StandardScaler(),
                           LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000))
rows = []
for fs, path in FEATS.items():
    df, cols = load_feats(path)
    X0, st, V = df[cols].values.astype(float), df["stage"].values, df["vendor"].astype(str).values
    Xz = center_z(X0, V)
    for held in CENTERS:
        tr = V != held; trc = [c for c in CENTERS if c != held]
        dm = DEEP[DEEP.held == held].set_index("filename")
        fn_te = df.loc[~tr, "filename"].values
        for tau in TAUS:
            keep = stable_cols(X0, st, V, trc, tau)
            if len(keep) < 2:
                rows.append(dict(feat=fs, held=held, tau=tau, n_sel=len(keep))); continue
            for mname, mf in [("rf", rf), ("logreg", lr)]:
                for task in TASKS:
                    p = score(mf().fit(Xz[tr][:, keep], label(st[tr], task)), Xz[~tr][:, keep])
                    y = label(st[~tr], task); d = dm.loc[fn_te, f"p_{task}"].values; fu = 0.5*d + 0.5*p
                    rec = dict(feat=fs, held=held, tau=tau, n_sel=len(keep), model=mname, task=task,
                               stable=auc(y, p), deep=auc(y, d), fusion=auc(y, fu))
                    if tau == PRIMARY_TAU:
                        I = boot(y)
                        lo, hi = ci([auc(y, p, i) for i in I]); rec["stable_ci"] = f"{lo:.2f}-{hi:.2f}"
                        lo, hi = ci([auc(y, fu, i) - auc(y, d, i) for i in I]); rec["fus_minus_deep_ci"] = f"{lo:+.3f},{hi:+.3f}"
                    rows.append(rec)
M = pd.DataFrame(rows); M.to_csv(f"{RES}/M_stable_selection.csv", index=False)
prim = M[M.tau == PRIMARY_TAU]
print(f"=== PRIMARY tau={PRIMARY_TAU} ===")
print(prim[["feat","held","model","task","n_sel","stable","stable_ci","deep","fusion","fus_minus_deep_ci"]].round(3).to_string(index=False))
print("\n=== sensitivity over tau (stable-feature AUC) ===")
print(M.dropna(subset=["model"]).pivot_table(index=["feat","model","task","tau"], columns="held", values="stable").round(3).to_string())

**Step 3.** CELL G: labeled examinations from the held out scanner (Sec. S5).

In [ ]:
# ===================== CELL 3: G (local-label budget curve) =====================
KS, REPS = [0, 5, 10, 20, 40], 10
rows = []
for fs, path in FEATS.items():
    df, cols = load_feats(path)
    X0, st, V, fn = df[cols].values.astype(float), df["stage"].values, df["vendor"].astype(str).values, df["filename"].values
    for vn in ["raw", "center-z"]:
        X = VARIANTS[vn](X0, V)
        for held in CENTERS:
            tgt, src = np.where(V == held)[0], np.where(V != held)[0]
            dmap = DEEP[DEEP.held == held].set_index("filename")
            for task in TASKS:
                ya = label(st, task)
                for k in KS:
                    for rep in range(REPS if k > 0 else 1):
                        r2 = np.random.default_rng(1000*rep + k)
                        for _ in range(50):
                            lab = r2.choice(tgt, k, replace=False) if k > 0 else np.array([], int)
                            test = np.setdiff1d(tgt, lab)
                            if len(np.unique(ya[test])) == 2: break
                        else:
                            continue
                        trn = np.r_[src, lab]
                        p = score(rf(300).fit(X[trn], ya[trn]), X[test])
                        d = dmap.loc[fn[test], f"p_{task}"].values; yt = ya[test]
                        rows.append(dict(feat=fs, variant=vn, held=held, task=task, k=k, rep=rep,
                                         rf=auc(yt, p), deep=auc(yt, d), fusion=auc(yt, 0.5*p + 0.5*d)))
Gd = pd.DataFrame(rows); Gd.to_csv(f"{RES}/G_label_budget.csv", index=False)
agg = Gd.groupby(["feat", "variant", "task", "held", "k"])[["rf", "deep", "fusion"]].agg(["mean", "std"]).round(3)
print(agg.to_string())

**Step 4.** CELL H: accuracy at three thresholds, calibration and four class decisions (Sec. S5).

In [ ]:
# ===================== CELL 4: H (accuracy at three thresholds, calibration, 4-class) =====================
def ece(y, p, b=10):
    bins = np.clip((p*b).astype(int), 0, b-1)
    return sum((bins == k).mean() * abs(y[bins == k].mean() - p[bins == k].mean()) for k in range(b) if (bins == k).any())
def oracle_acc(y, p):
    return max(accuracy_score(y, (p >= t).astype(int)) for t in np.r_[np.unique(p), np.inf])

C = pd.read_csv(f"{RES}/B_loco_cases.csv")
rows = []
for (fs, vn, held), m in C.groupby(["feat", "variant", "held"], sort=False):
    for task in TASKS:
        y = label(m.stage, task)
        pi = label(DEEP[DEEP.held != held].stage, task).mean()
        for name, p in [("deep", m[f"p_{task}"].values), ("rf", m[f"rf_{task}"].values),
                        ("fusion", 0.5*m[f"p_{task}"].values + 0.5*m[f"rf_{task}"].values)]:
            t_prev = np.quantile(p, 1 - pi)
            rows.append(dict(feat=fs, variant=vn, held=held, task=task, stream=name,
                             acc_0_5=accuracy_score(y, (p >= 0.5).astype(int)),
                             acc_prev=accuracy_score(y, (p >= t_prev).astype(int)),
                             acc_oracle=oracle_acc(y, p), ece=ece(y, p), brier=brier_score_loss(y, p)))
H = pd.DataFrame(rows); H.to_csv(f"{RES}/H_accuracy_calibration.csv", index=False)
for (fs, vn), g in H.groupby(["feat", "variant"], sort=False):
    print(f"\n=== {fs} | {vn} ===\n{g.drop(columns=['feat','variant']).round(3).to_string(index=False)}")
rows = []
for held in CENTERS:
    g = DEEP[DEEP.held == held]; D = g[["d1", "d2", "d3", "d4"]].values; y4 = g.stage.values - 1
    rows.append(dict(held=held, acc_4class=(D.argmax(1) == y4).mean(),
                     macro_auc_4class=roc_auc_score(y4, D, multi_class="ovr", labels=[0, 1, 2, 3])))
q = pd.DataFrame(rows); print("\n=== deep, four-class (challenge task) ===\n", q.round(3).to_string(index=False))
q.to_csv(f"{RES}/H_deep_4class.csv", index=False)

**Step 5.** CELL N: selective prediction (not reported in the paper).

In [ ]:
# ===================== CELL 5: N (selective prediction) =====================
COVER = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]
rows = []
for held in CENTERS:
    g = dp[dp.held == held]
    for task in TASKS:
        piv = g.pivot_table(index="filename", columns="seed", values=f"p_{task}")
        p = piv.mean(1).values; sd = piv.std(1).values
        st_ = DEEP[DEEP.held == held].set_index("filename").loc[piv.index, "stage"].values
        y = label(st_, task)
        pi = label(DEEP[DEEP.held != held].stage, task).mean()
        t = np.quantile(p, 1 - pi)
        for uname, conf in [("margin", np.abs(p - t)), ("seed-agreement", -sd)]:
            order = np.argsort(-conf)
            for c in COVER:
                k = max(1, int(round(c*len(y)))); keep = order[:k]
                rows.append(dict(held=held, task=task, score=uname, coverage=c, n=k,
                                 acc=accuracy_score(y[keep], (p[keep] >= t).astype(int)),
                                 auc=auc(y[keep], p[keep]) if len(np.unique(y[keep])) == 2 else np.nan))
N = pd.DataFrame(rows); N.to_csv(f"{RES}/N_selective_prediction.csv", index=False)
print(N.pivot_table(index=["task","score","held"], columns="coverage", values="acc").round(3).to_string())

**Step 6.** CELL C: pooled versus held out evaluation on B2 (Table 2).

In [ ]:
# ===================== CELL 6: C (leaky vs clean protocol on B2) =====================
b2deep = DEEP[DEEP.held == "B2"]
rows = []
for name, path in {"old (bug)": f"{CKPT}/lifs_stad_enriched.csv", **FEATS}.items():
    df, cols = load_feats(path)
    X, st, V = df[cols].values.astype(float), df["stage"].values, df["vendor"].astype(str).values
    b2 = V == "B2"
    for proto in ["leaky", "held-out"]:
        rec = dict(features=name, protocol=proto)
        for task in TASKS:
            ya = label(st, task)
            if proto == "leaky":
                p = cross_val_predict(rf(), X, ya, cv=StratifiedKFold(5, shuffle=True, random_state=42),
                                      method="predict_proba")[:, 1][b2]
            else:
                p = score(rf().fit(X[~b2], ya[~b2]), X[b2])
            t = pd.DataFrame({"filename": df.loc[b2, "filename"].values, "rf": p}).merge(b2deep, on="filename")
            y = label(t.stage, task)
            rec[f"rf_{task}"] = auc(y, t.rf.values)
            rec[f"fusion_{task}"] = auc(y, 0.5*t.rf.values + 0.5*t[f"p_{task}"].values)
        rows.append(rec)
tab = pd.DataFrame(rows)
print(f"deep, held-out B2 (unchanged): sub {auc(label(b2deep.stage,'sub'), b2deep.p_sub.values):.3f}  "
      f"cir {auc(label(b2deep.stage,'cir'), b2deep.p_cir.values):.3f}")
print(tab.round(3).to_string(index=False))
tab.to_csv(f"{RES}/C_leaky_vs_clean.csv", index=False)

**Step 7.** CELL D: how well the features predict the scanner. Run it once, before CELL J, and ignore its message asking for a rerun. If `paper_results/deep_embeddings.npz` already exists when CELL D runs, it adds a deep embedding row that CELL 10 cannot read; delete that row from `D_vendor_predictability.csv`.

In [ ]:
# ===================== CELL 7: D (how strongly features encode the scanner) =====================
def vendor_test(X, V):
    P = cross_val_predict(rf(), X, V, cv=StratifiedKFold(5, shuffle=True, random_state=42), method="predict_proba")
    cl = np.unique(V)
    return roc_auc_score(V, P, multi_class="ovr", labels=cl), accuracy_score(V, cl[P.argmax(1)])
rows = []
for fs, path in FEATS.items():
    df, cols = load_feats(path)
    X0, V, st = df[cols].values.astype(float), df["vendor"].astype(str).values, df["stage"].values
    for vn, vfn in VARIANTS.items():
        X = vfn(X0, V); va, vacc = vendor_test(X, V)
        cir = auc(label(st, "cir"), cross_val_predict(rf(), X, label(st, "cir"),
                  cv=StratifiedKFold(5, shuffle=True, random_state=42), method="predict_proba")[:, 1])
        rows.append(dict(source=fs, variant=vn, vendor_auc=va, vendor_acc=vacc, cirrhosis_auc_pooled=cir))
emb = f"{RES}/deep_embeddings.npz"
if os.path.exists(emb):
    e = np.load(emb, allow_pickle=True); va, vacc = vendor_test(e["emb"], e["vendor"].astype(str))
    rows.append(dict(source="deep embedding (387-model)", variant="raw", vendor_auc=va, vendor_acc=vacc))
else:
    print("deep embeddings not ready yet (from CELL 11); rerun this cell after it for that row")
t = pd.DataFrame(rows); print(t.round(3).to_string(index=False)); t.to_csv(f"{RES}/D_vendor_predictability.csv", index=False)
print(f"chance accuracy (majority vendor): {pd.Series(V).value_counts(normalize=True).max():.3f}")

**Step 8.** CELL E: classical learners under leave one scanner out (Table S2).

In [ ]:
# ===================== CELL 8: E (classical baselines under LOCO) =====================
pre = lambda: [SimpleImputer(strategy="median"), FunctionTransformer(np.nan_to_num), StandardScaler()]
MODELS = {
    "logreg":  lambda: make_pipeline(*pre(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000)),
    "svm-lin": lambda: make_pipeline(*pre(), SVC(kernel="linear", C=0.1, class_weight="balanced")),
    "svm-rbf": lambda: make_pipeline(*pre(), SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced")),
    "gboost":  lambda: make_pipeline(FunctionTransformer(lambda a: np.where(np.isfinite(a), a, np.nan)),
                                     HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                                                    class_weight="balanced", random_state=42)),
    "rf":      rf,
}
rows = []
for fs, path in FEATS.items():
    df, cols = load_feats(path)
    X0, st, V = df[cols].values.astype(float), df["stage"].values, df["vendor"].astype(str).values
    for vn, vfn in VARIANTS.items():
        X = vfn(X0, V)
        for mn, mf in MODELS.items():
            for held in CENTERS:
                tr = V != held
                for task in TASKS:
                    p = score(mf().fit(X[tr], label(st[tr], task)), X[~tr])
                    rows.append(dict(feat=fs, variant=vn, model=mn, held=held, task=task, auc=auc(label(st[~tr], task), p)))
E = pd.DataFrame(rows); E.to_csv(f"{RES}/E_baselines.csv", index=False)
for (fs, vn), g in E.groupby(["feat", "variant"], sort=False):
    pv = g.pivot_table(index="model", columns=["task", "held"], values="auc"); pv["mean"] = pv.mean(1)
    print(f"\n=== {fs} | {vn} ===\n{pv.round(3).to_string()}")

**Step 9.** CELL F: feature groups and series (Sec. S4).

In [ ]:
# ===================== CELL 9: F (feature groups, per-phase drop, missing modality) =====================
def phase_of(c):
    for ph in PHASES:
        if c.startswith(ph + "_"): return ph, c[len(ph)+1:]
    return None, c
def groups(cols):
    g = {"shape": [c for c in cols if c.startswith("shape_")], "enriched": [c for c in cols if c in ENR],
         "first-order": [], "texture": [], "directional": []}
    for c in cols:
        ph, s = phase_of(c)
        if ph is None: continue
        if s in FO: g["first-order"].append(c)
        elif s in TX: g["texture"].append(c)
        elif s in DR: g["directional"].append(c)
    return {k: v for k, v in g.items() if v}
def loco_auc(X, st, V, cols_idx):
    out = {}
    for held in CENTERS:
        tr = V != held
        for task in TASKS:
            p = score(rf().fit(X[tr][:, cols_idx], label(st[tr], task)), X[~tr][:, cols_idx])
            out[(task, held)] = auc(label(st[~tr], task), p)
    return out
rows = []
for fs, path in FEATS.items():
    df, cols = load_feats(path)
    X0, st, V = df[cols].values.astype(float), df["stage"].values, df["vendor"].astype(str).values
    for vn in ["raw", "center-z"]:
        X = VARIANTS[vn](X0, V)
        for gname, gcols in groups(cols).items():
            idx = [cols.index(c) for c in gcols]
            for (task, held), a in loco_auc(X, st, V, idx).items():
                rows.append(dict(feat=fs, variant=vn, analysis="group only", what=gname, n_feat=len(idx), task=task, held=held, auc=a))
        if fs == "fixed7":
            for ph in PHASES:
                idx = [i for i, c in enumerate(cols) if not c.startswith(ph + "_")]
                for (task, held), a in loco_auc(X, st, V, idx).items():
                    rows.append(dict(feat=fs, variant=vn, analysis="drop phase", what=ph, n_feat=len(idx), task=task, held=held, auc=a))
    if fs == "fixed7":
        for held in CENTERS:
            tr = V != held
            for task in TASKS:
                mdl = rf().fit(X0[tr], label(st[tr], task)); yt = label(st[~tr], task)
                rows.append(dict(feat=fs, variant="raw", analysis="missing at test", what="none", n_feat=len(cols),
                                 task=task, held=held, auc=auc(yt, score(mdl, X0[~tr]))))
                for ph in PHASES:
                    Xt = X0[~tr].copy(); Xt[:, [i for i, c in enumerate(cols) if c.startswith(ph + "_")]] = np.nan
                    rows.append(dict(feat=fs, variant="raw", analysis="missing at test", what=ph, n_feat=len(cols),
                                     task=task, held=held, auc=auc(yt, score(mdl, Xt))))
Fd = pd.DataFrame(rows); Fd.to_csv(f"{RES}/F_feature_ablations.csv", index=False)
for (fs, vn, an), g in Fd.groupby(["feat", "variant", "analysis"], sort=False):
    pv = g.pivot_table(index="what", columns=["task", "held"], values="auc"); pv["mean"] = pv.mean(1)
    print(f"\n=== {fs} | {vn} | {an} ===\n{pv.round(3).to_string()}")

**Step 10.** CELL I: figures from the saved results.

In [ ]:
# ===================== CELL 10: I (figures) =====================
import matplotlib.pyplot as plt
S = pd.read_csv(f"{RES}/B_loco_summary.csv"); Gd = pd.read_csv(f"{RES}/G_label_budget.csv")
FEAT = "fixed7" if "fixed7" in S.feat.unique() else "ged4"
TT = {"sub": "Substantial fibrosis (S2+)", "cir": "Cirrhosis (S4)"}
for vn in ["raw", "center-z", "center-rank"]:
    g = S[(S.feat == FEAT) & (S.variant == vn)]
    fig, axes = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
    for ax, task in zip(axes, TASKS):
        t = g[g.task == task].set_index("held").loc[CENTERS]; x = np.arange(3); w = 0.26
        for j, (k, lab) in enumerate([("deep", "Deep"), ("rf", "Radiomic RF"), ("fusion", "Fusion")]):
            v = t[k].values
            ax.bar(x + (j-1)*w, v, w, yerr=[v - t[f"{k}_lo"].values, t[f"{k}_hi"].values - v], capsize=3, label=lab)
        ax.axhline(0.5, color="k", lw=0.8, ls="--"); ax.set_ylim(0, 1); ax.set_title(TT[task])
        ax.set_xticks(x); ax.set_xticklabels([f"held-out {c}" for c in CENTERS])
    axes[0].set_ylabel("AUC"); axes[1].legend(frameon=False, fontsize=8)
    plt.tight_layout(); plt.savefig(f"{RES}/fig_loco_{FEAT}_{vn}.pdf"); plt.savefig(f"{RES}/fig_loco_{FEAT}_{vn}.png", dpi=200); plt.show()
fig, axes = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
for ax, task in zip(axes, TASKS):
    for held in CENTERS:
        g = Gd[(Gd.feat == FEAT) & (Gd.variant == "raw") & (Gd.task == task) & (Gd.held == held)].groupby("k")
        ax.errorbar(g.rf.mean().index, g.rf.mean(), yerr=g.rf.std().fillna(0), marker="o", capsize=3, label=f"RF, {held}")
    ax.axhline(0.5, color="k", lw=0.8, ls="--"); ax.set_xlabel("labeled cases from the new center"); ax.set_title(TT[task])
axes[0].set_ylabel("AUC"); axes[1].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(f"{RES}/fig_label_budget_{FEAT}.pdf"); plt.savefig(f"{RES}/fig_label_budget_{FEAT}.png", dpi=200); plt.show()
print("figures saved to", RES)

**Step 11.** CELL J (GPU): five fold cross validation of the deep stream in distribution (seed 42), and embeddings of every examination from the seed 42 deep stream of notebook 3 (trained on A and B1).

In [ ]:
# ===================== CELL 11 (GPU SESSION): J (deep in-distribution CV + embeddings) =====================
# Skip if already run: check that paper_results/id_cv_deep_preds.csv exists.
!pip install -q timm 2>&1 | tail -1
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, glob, os, time, random
import torch, torch.nn as nn, torch.nn.functional as F, timm
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
torch.backends.cudnn.benchmark = True
CKPT  = "/content/drive/MyDrive/CARE2026_ckpts"
HIRES = f"{CKPT}/lifs_ged4_hires_train_v2"
RES   = f"{CKPT}/paper_results"; os.makedirs(RES, exist_ok=True)
OUT   = f"{CKPT}/lifs_id_cv"; os.makedirs(OUT, exist_ok=True)
files = sorted(glob.glob(f"{HIRES}/*.npz"))
VEND, STAGE = {}, {}
for f in files:
    d = np.load(f); VEND[f] = str(d["vendor"]); STAGE[f] = int(d["stage"])
class DS(Dataset):
    def __init__(s, fl): s.fl = fl
    def __len__(s): return len(s.fl)
    def __getitem__(s, i):
        x = np.load(s.fl[i])["ged4"].astype(np.float32); y = STAGE[s.fl[i]] - 1
        if random.random() < .5: x = x[:, :, ::-1]
        if random.random() < .5: x = x[:, ::-1, :]
        return torch.from_numpy(x.copy()), y
class SliceAttnNet(nn.Module):
    def __init__(s, n=4, pretrained=True):
        super().__init__()
        s.enc = timm.create_model("convnext_tiny", pretrained=pretrained, in_chans=1, num_classes=0, global_pool="avg")
        f = s.enc.num_features
        s.attn = nn.Sequential(nn.Linear(f, 128), nn.Tanh(), nn.Linear(128, 1))
        s.head = nn.Sequential(nn.Dropout(0.5), nn.Linear(f, n))
    def embed(s, x):
        B, S, H, W = x.shape
        fe = s.enc(x.reshape(B*S, 1, H, W)).reshape(B, S, -1)
        a = torch.softmax(s.attn(fe).squeeze(-1), 1)
        return (fe*a.unsqueeze(-1)).sum(1)
    def forward(s, x): return s.head(s.embed(x))
def train(tr_files, seed, epochs=40):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    cnt = Counter(STAGE[f]-1 for f in tr_files); tot = sum(cnt.values())
    cw = torch.tensor([tot/(4*max(cnt[c], 1)) for c in range(4)], dtype=torch.float32).cuda()
    dl = DataLoader(DS(tr_files), batch_size=4, shuffle=True, num_workers=2, drop_last=True)
    m = SliceAttnNet().cuda()
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=1e-5)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs); sc = torch.amp.GradScaler('cuda'); m.train()
    for ep in range(epochs):
        t0 = time.time()
        for x, y in dl:
            x, y = x.cuda(), y.cuda(); opt.zero_grad()
            with torch.amp.autocast('cuda'):
                ce = F.cross_entropy(m(x), y, weight=cw, reduction="none")
                loss = torch.topk(ce, max(1, int(0.8*ce.numel()))).values.mean()
            sc.scale(loss).backward(); sc.step(opt); sc.update()
        sch.step()
        if ep % 10 == 0 or ep == epochs-1: print(f"    ep{ep:02d} ({time.time()-t0:.0f}s)")
    return m.eval()
@torch.no_grad()
def predict(m, fl):
    out = []
    for f in fl:
        x = np.load(f)["ged4"].astype(np.float32)
        P = [F.softmax(m(torch.from_numpy(v.copy())[None].cuda().float()), 1)[0].cpu().numpy()
             for v in (x, x[:, :, ::-1], x[:, ::-1, :], x[:, ::-1, ::-1])]
        out.append(np.mean(P, 0))
    return np.array(out)
ep = f"{RES}/deep_embeddings.npz"
if not os.path.exists(ep):
    m = SliceAttnNet(pretrained=False).cuda()
    m.load_state_dict(torch.load(f"{CKPT}/lifs_ged4_v2_model/ged4_v2_seed42.pth")); m.eval()
    E = []
    with torch.no_grad():
        for f in files:
            x = torch.from_numpy(np.load(f)["ged4"].astype(np.float32))[None].cuda()
            E.append(m.embed(x).float().cpu().numpy()[0])
    np.savez(ep, filename=np.array([os.path.basename(f)[:-4] for f in files]),
             vendor=np.array([VEND[f] for f in files]), stage=np.array([STAGE[f] for f in files]), emb=np.array(E))
    del m; torch.cuda.empty_cache(); print("embeddings saved")
strat = [f"{VEND[f]}_{STAGE[f]}" for f in files]
rows = []
for k, (tr, te) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(files, strat)):
    pp = f"{OUT}/preds_fold{k}.npy"; tef = [files[i] for i in te]
    if os.path.exists(pp):
        P = np.load(pp); print(f"fold {k}: cached")
    else:
        mp = f"{OUT}/model_fold{k}.pth"
        if os.path.exists(mp):
            m = SliceAttnNet(pretrained=False).cuda(); m.load_state_dict(torch.load(mp)); m.eval()
        else:
            print(f"fold {k}: training on {len(tr)}"); m = train([files[i] for i in tr], 42); torch.save(m.state_dict(), mp)
        P = predict(m, tef); np.save(pp, P); del m; torch.cuda.empty_cache()
    for f, p in zip(tef, P):
        rows.append(dict(filename=os.path.basename(f)[:-4], vendor=VEND[f], stage=STAGE[f], fold=k,
                         d1=p[0], d2=p[1], d3=p[2], d4=p[3]))
idd = pd.DataFrame(rows); idd.to_csv(f"{RES}/id_cv_deep_preds.csv", index=False)
for v in ["A", "B1", "B2"]:
    g = idd[idd.vendor == v]
    print(f"{v}: sub {roc_auc_score(g.stage >= 2, 1-g.d1):.3f}  cir {roc_auc_score(g.stage >= 4, g.d4):.3f}")

**Step 12.** Controls on the feature sets (`CTRL_feature_sets.csv`, Sec. S4).

In [ ]:
# === FINAL CONTROLS: phases vs enriched features, deep-embedding scanner test, 4-class confusion ===
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, accuracy_score
pd.set_option("display.width", 220)

CKPT = "/content/drive/MyDrive/CARE2026_ckpts"; RES = f"{CKPT}/paper_results"
ENR = ['mc_surface','convex_vol','surf_to_hull','compactness2','roi_skew2','roi_kurt2','roi_p10','roi_p90','roi_iqr2']
df = pd.read_csv(f"{CKPT}/lifs_stad_fixed.csv")
st, V = df.stage.values, df.vendor.astype(str).values
base = [c for c in df.columns if c not in ("filename", "vendor", "stage") and c not in ENR]
g4 = [c for c in base if c.startswith(("GED4_", "shape_"))]
SETS = {"7 phases + enriched (198)": base + ENR, "7 phases, no enriched (189)": base,
        "GED4 + enriched (42)": g4 + ENR, "GED4, no enriched (33)": g4, "enriched only (9)": ENR}

rf = lambda: make_pipeline(SimpleImputer(strategy="median"), FunctionTransformer(np.nan_to_num),
                           RandomForestClassifier(n_estimators=600, min_samples_leaf=2, max_features="sqrt",
                                                  class_weight="balanced", n_jobs=-1, random_state=42))
rng = np.random.default_rng(0)
def boot_ci(y, p, n=2000):
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    v = [roc_auc_score(y[i], p[i]) for i in (np.r_[rng.choice(pos, len(pos)), rng.choice(neg, len(neg))] for _ in range(n))]
    return np.percentile(v, [2.5, 97.5])

rows = []
for name, cols in SETS.items():
    X = df[cols].values.astype(float)
    for held in ["A", "B1", "B2"]:
        tr = V != held
        for task, thr in [("sub", 2), ("cir", 4)]:
            y = (st[~tr] >= thr).astype(int)
            p = rf().fit(X[tr], st[tr] >= thr).predict_proba(X[~tr])[:, 1]
            lo, hi = boot_ci(y, p)
            rows.append(dict(features=name, held=held, task=task, auc=roc_auc_score(y, p),
                             ci=f"{lo:.2f}-{hi:.2f}", verdict="above" if lo > 0.5 else ("BELOW" if hi < 0.5 else "~0.5")))
T = pd.DataFrame(rows); T.to_csv(f"{RES}/CTRL_feature_sets.csv", index=False)
order = list(SETS)
a = T.pivot(index="features", columns=["task", "held"], values="auc").loc[order]; a["mean"] = a.mean(1)
print("=== RF AUC, leave-one-center-out ===\n" + a.round(3).to_string())
print("\n=== 95% CI ===\n" + T.pivot(index="features", columns=["task", "held"], values="ci").loc[order].to_string())
print("\n=== vs chance ===\n" + T.pivot(index="features", columns=["task", "held"], values="verdict").loc[order].to_string())
print("\nread: if '7 phases, no enriched' is fine but 'GED4, no enriched' still fails on A -> the PHASES matter.")
print("      if 'GED4, no enriched' is fine too -> the failure was the ENRICHED features, not the phase count.")

# ---- deep embedding: can the deep model's features identify the scanner too? ----
e = np.load(f"{RES}/deep_embeddings.npz", allow_pickle=True); Ve = e["vendor"].astype(str); cl = np.unique(Ve)
P = cross_val_predict(RandomForestClassifier(n_estimators=600, class_weight="balanced", n_jobs=-1, random_state=42),
                      e["emb"], Ve, cv=StratifiedKFold(5, shuffle=True, random_state=42), method="predict_proba")
print(f"\n=== deep embedding (387-model) -> scanner: AUC {roc_auc_score(Ve, P, multi_class='ovr', labels=cl):.3f}, "
      f"accuracy {accuracy_score(Ve, cl[P.argmax(1)]):.3f}  (radiomics were 0.99-1.00) ===")

# ---- deep four-class confusion matrices (explains the low challenge accuracy) ----
dp = pd.read_csv(f"{CKPT}/lifs_loco_deep/loco_deep_preds.csv")
E4 = dp.groupby(["filename", "held", "stage"], as_index=False)[["d1", "d2", "d3", "d4"]].mean()
for held in ["A", "B1", "B2"]:
    g = E4[E4.held == held]
    pred = g[["d1", "d2", "d3", "d4"]].values.argmax(1) + 1
    cm = pd.crosstab(pd.Series(g.stage.values, name="true stage"), pd.Series(pred, name="predicted"))
    print(f"\n=== deep 4-class confusion, held-out {held} ===\n{cm.to_string()}")

**Step 13.** CELL O: four class decisions (Sec. S5).

In [ ]:
# ===================== CELL O: four-class, deployed construction vs direct argmax =====================
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, cohen_kappa_score
CKPT = "/content/drive/MyDrive/CARE2026_ckpts"; RES = f"{CKPT}/paper_results"
C = pd.read_csv(f"{RES}/B_loco_cases.csv")
C = C[(C.feat == "fixed7") & (C.variant == "raw")].reset_index(drop=True)

def construct(d, ps, pc):                       # as in deployed cell 154
    pc = np.minimum(pc, ps); mid = ps - pc       # enforce P(>=S4) <= P(>=S2)
    r = d[:, 1] / (d[:, 1] + d[:, 2] + 1e-8)     # split S2/S3 mass by the deep S2:S3 ratio
    return np.c_[1 - ps, mid * r, mid * (1 - r), pc]

rng = np.random.default_rng(0); rows = []
for held in ["A", "B1", "B2"]:
    g = C[C.held == held]; y = g.stage.values; D = g[["d1", "d2", "d3", "d4"]].values
    ds, dc, rs, rc = 1 - g.d1.values, g.d4.values, g.rf_sub.values, g.rf_cir.values
    M = {"deep argmax": D,
         "constructed, equal fusion": construct(D, 0.5*ds + 0.5*rs, 0.5*dc + 0.5*rc),
         "constructed, as shipped (sub 0.35 deep)": construct(D, 0.35*ds + 0.65*rs, 0.5*dc + 0.5*rc)}
    base = D.argmax(1) + 1
    idx = [rng.integers(0, len(y), len(y)) for _ in range(2000)]
    for name, P in M.items():
        pred = P.argmax(1) + 1; acc = accuracy_score(y, pred)
        lo, hi = np.percentile([accuracy_score(y[i], pred[i]) - accuracy_score(y[i], base[i]) for i in idx], [2.5, 97.5])
        rows.append(dict(held=held, method=name, acc=acc, qwk=cohen_kappa_score(y, pred, weights="quadratic"),
                         macro_auc=roc_auc_score(y - 1, P, multi_class="ovr", labels=[0, 1, 2, 3]),
                         acc_vs_argmax=f"{acc - accuracy_score(y, base):+.3f} [{lo:+.3f},{hi:+.3f}]",
                         pred_S1_S2_S3_S4=str(np.bincount(pred - 1, minlength=4).tolist())))
O = pd.DataFrame(rows); O.to_csv(f"{RES}/O_four_class.csv", index=False)
print(O.round(3).to_string(index=False))
print("\ntrue S1_S2_S3_S4:", {h: np.bincount(C[C.held == h].stage - 1, minlength=4).tolist() for h in ["A", "B1", "B2"]})

**Step 14.** CELL K: in distribution table. This self contained version, run after CELL J, wrote `K_in_distribution.csv` in our run.

In [ ]:
# ===================== CELL K (self-contained): in-distribution table, run after J =====================
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, os, warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
CKPT = "/content/drive/MyDrive/CARE2026_ckpts"; RES = f"{CKPT}/paper_results"
FEATS = {"ged4": f"{CKPT}/lifs_stad_ged4_clean.csv", "fixed7": f"{CKPT}/lifs_stad_fixed.csv"}
TASKS = {"sub": 2, "cir": 4}; CENTERS = ["A", "B1", "B2"]
label = lambda s, t: (np.asarray(s) >= TASKS[t]).astype(int)
rng = np.random.default_rng(0)
def boot(y, n=2000):
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    return [np.r_[rng.choice(pos, len(pos)), rng.choice(neg, len(neg))] for _ in range(n)]
auc = lambda y, p, i=None: roc_auc_score(y, p) if i is None else roc_auc_score(y[i], p[i])
rf = lambda: make_pipeline(SimpleImputer(strategy="median"), FunctionTransformer(np.nan_to_num),
                           RandomForestClassifier(n_estimators=600, min_samples_leaf=2, max_features="sqrt",
                                                  class_weight="balanced", n_jobs=-1, random_state=42))
idd = pd.read_csv(f"{RES}/id_cv_deep_preds.csv"); idd["p_sub"] = 1 - idd.d1; idd["p_cir"] = idd.d4
rows = []
for fs, path in FEATS.items():
    df = pd.read_csv(path)
    cols = [c for c in df.columns if c not in ("filename", "vendor", "stage")]
    m = idd[["filename", "vendor", "stage", "fold", "p_sub", "p_cir"]].copy()
    X = df.set_index("filename").loc[m.filename, cols].values.astype(float)
    for task in TASKS:
        y = label(m.stage, task); r = np.zeros(len(m))
        for k in sorted(m.fold.unique()):
            te = m.fold.values == k
            r[te] = rf().fit(X[~te], y[~te]).predict_proba(X[te])[:, 1]
        m[f"rf_{task}"] = r
    for scope in ["pooled"] + CENTERS:
        g = m if scope == "pooled" else m[m.vendor.astype(str) == scope]
        for task in TASKS:
            y = label(g.stage, task); I = boot(y)
            d, rr = g[f"p_{task}"].values, g[f"rf_{task}"].values; fu = 0.5*d + 0.5*rr
            rec = dict(feat=fs, scope=scope, task=task, n=len(y), n_neg=int((y == 0).sum()))
            for name, p in [("deep", d), ("rf", rr), ("fusion", fu)]:
                lo, hi = np.percentile([auc(y, p, i) for i in I], [2.5, 97.5])
                rec[name] = auc(y, p); rec[f"{name}_ci"] = f"{lo:.2f}-{hi:.2f}"
            rows.append(rec)
K = pd.DataFrame(rows); K.to_csv(f"{RES}/K_in_distribution.csv", index=False)
print(K.round(3).to_string(index=False))

**Step 15.** CELL P (GPU): frozen DINOv2 probe under leave one scanner out (Table 3).

In [ ]:
# ===================== CELL P (GPU, OPTIONAL): frozen foundation model, DINOv2 ViT-B/14, LOCO =====================
!pip install -q timm 2>&1 | tail -1
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, glob, os, time, warnings
import torch, torch.nn.functional as F, timm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, accuracy_score
warnings.filterwarnings("ignore")
CKPT = "/content/drive/MyDrive/CARE2026_ckpts"; RES = f"{CKPT}/paper_results"
HIRES = f"{CKPT}/lifs_ged4_hires_train_v2"; EMB = f"{RES}/fm_dinov2_embeddings.npz"
CENTERS, TASKS = ["A", "B1", "B2"], {"sub": 2, "cir": 4}

if not os.path.exists(EMB):
    try:
        fm = timm.create_model("vit_base_patch14_dinov2.lvd142m", pretrained=True, num_classes=0, img_size=224); src = "timm dinov2 vitb14"
    except Exception as e:
        print("timm load failed, using torch.hub:", str(e)[:80])
        fm = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14"); src = "hub dinov2_vitb14"
    fm = fm.cuda().eval()
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).cuda(); std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).cuda()
    files = sorted(glob.glob(f"{HIRES}/*.npz")); meta, E = [], []; t0 = time.time()
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for i, f in enumerate(files, 1):
            d = np.load(f); x = torch.from_numpy(d["ged4"].astype(np.float32)).cuda()        # (24,256,256), z-scored
            x = ((x.clamp(-3, 3) + 3) / 6).unsqueeze(1)                                         # -> [0,1]
            x = F.interpolate(x, size=(224, 224), mode="bilinear", align_corners=False).repeat(1, 3, 1, 1)
            E.append(fm((x - mean) / std).float().mean(0).cpu().numpy())
            meta.append((os.path.basename(f)[:-4], str(d["vendor"]), int(d["stage"])))
            if i % 100 == 0: print(f"  {i}/{len(files)} ({time.time()-t0:.0f}s)")
    fn, ven, stg = zip(*meta)
    np.savez(EMB, filename=np.array(fn), vendor=np.array(ven), stage=np.array(stg), emb=np.array(E), source=src)

e = np.load(EMB, allow_pickle=True)
X0, V, st, fn = e["emb"], e["vendor"].astype(str), e["stage"].astype(int), e["filename"].astype(str)
print("embeddings", X0.shape, "| source:", e["source"])

def center_z(X, V):
    Z = X.copy().astype(float)
    for v in np.unique(V):
        m = V == v; sd = X[m].std(0); sd[sd < 1e-8] = 1; Z[m] = (X[m] - X[m].mean(0)) / sd
    return Z
lr = lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000))
dp = pd.read_csv(f"{CKPT}/lifs_loco_deep/loco_deep_preds.csv"); dp["p_sub"] = 1 - dp.d1; dp["p_cir"] = dp.d4
DEEP = dp.groupby(["filename", "held"], as_index=False)[["p_sub", "p_cir"]].mean()
rng = np.random.default_rng(0)
def boot(y, n=2000):
    pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]
    return [np.r_[rng.choice(pos, len(pos)), rng.choice(neg, len(neg))] for _ in range(n)]
auc = lambda y, p, i=None: roc_auc_score(y, p) if i is None else roc_auc_score(y[i], p[i])
ci = lambda v: np.percentile(v, [2.5, 97.5])

rows = []
for vn, X in [("raw", X0), ("center-z", center_z(X0, V))]:
    for held in CENTERS:
        tr = V != held; dm = DEEP[DEEP.held == held].set_index("filename")
        for task, thr in TASKS.items():
            y = (st[~tr] >= thr).astype(int); I = boot(y)
            p = lr().fit(X[tr], st[tr] >= thr).predict_proba(X[~tr])[:, 1]
            d = dm.loc[fn[~tr], f"p_{task}"].values; fu = 0.5*p + 0.5*d
            rec = dict(variant=vn, held=held, task=task, n_neg=int((y == 0).sum()))
            for name, q in [("fm", p), ("deep", d), ("fm+deep", fu)]:
                lo, hi = ci([auc(y, q, i) for i in I]); rec[name] = auc(y, q); rec[f"{name}_ci"] = f"{lo:.2f}-{hi:.2f}"
            lo, hi = ci([auc(y, p, i) - auc(y, d, i) for i in I])
            rec["fm_minus_deep"] = f"{auc(y, p) - auc(y, d):+.3f} [{lo:+.3f},{hi:+.3f}]"
            rows.append(rec)
P = pd.DataFrame(rows); P.to_csv(f"{RES}/P_foundation_model.csv", index=False)
print(P.round(3).to_string(index=False))
print("\nmean over 6 pairs:\n" + P.groupby("variant")[["fm", "deep", "fm+deep"]].mean().round(3).to_string())

cl = np.unique(V)
Pv = cross_val_predict(RandomForestClassifier(n_estimators=600, class_weight="balanced", n_jobs=-1, random_state=42),
                       X0, V, cv=StratifiedKFold(5, shuffle=True, random_state=42), method="predict_proba")
print(f"\nfoundation-model embedding -> scanner: AUC {roc_auc_score(V, Pv, multi_class='ovr', labels=cl):.3f}, "
      f"accuracy {accuracy_score(V, cl[Pv.argmax(1)]):.3f}   (radiomics 0.99-1.00, trained deep 0.935)")

**Step 16.** CELL Q: sensitivity to the fusion weight (Sec. S1).

In [ ]:
# === Q. Fusion weight & rule sensitivity, fixed7 | raw, LOCO (CPU, ~1 min, saved predictions only) ===
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
RES = "/content/drive/MyDrive/CARE2026_ckpts/paper_results"
C = pd.read_csv(f"{RES}/B_loco_cases.csv"); C = C[(C.feat == "fixed7") & (C.variant == "raw")]
W = [round(w, 1) for w in np.arange(0, 1.01, 0.1)]
rows = []
for held in ["A", "B1", "B2"]:
    g = C[C.held == held]
    for task, thr in [("sub", 2), ("cir", 4)]:
        y = (g.stage.values >= thr).astype(int); d, r = g[f"p_{task}"].values, g[f"rf_{task}"].values
        rec = {"pair": f"{held} {task}"}
        for w in W: rec[w] = roc_auc_score(y, w*r + (1-w)*d)
        rec["rank-avg"] = roc_auc_score(y, 0.5*rankdata(d) + 0.5*rankdata(r))
        rows.append(rec)
T = pd.DataFrame(rows).set_index("pair"); T.to_csv(f"{RES}/Q_fusion_sensitivity.csv")
print("AUC by RF weight (0 = deep only, 1 = RF only), plus rank averaging:\n" + T.round(3).to_string())
core = T.drop(index="B2 sub")                                   # 5 negatives, unreliable
regret = (-T).add(T[W].max(axis=1), axis=0).max()               # worst loss vs the best weight for each pair
S = pd.DataFrame({"mean_all6": T.mean(), "mean_excl_B2sub": core.mean(), "worst_pair": T.min(), "max_regret": regret}).T
print("\nsummary:\n" + S.round(3).to_string())
print("\nbest weight by mean (excl. B2 sub):", core[W].mean().idxmax(), "| by worst case:", T[W].min().idxmax())
print("NOTE: 0.5 was fixed in advance. Report this as sensitivity only; never switch to the best weight.")

**Step 17.** CELL U: ComBat harmonization of the radiomic features (Tables 3 and S2).

In [ ]:
# ===================== CELL U (CPU, ~5 min): ComBat harmonization of the radiomic features =====================
!pip install -q inmoose 2>&1 | tail -1
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, warnings
from inmoose.pycombat import pycombat_norm
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
CKPT = "/content/drive/MyDrive/CARE2026_ckpts"; RES = f"{CKPT}/paper_results"
CENTERS, TASKS = ["A", "B1", "B2"], {"sub": 2, "cir": 4}

df = pd.read_csv(f"{CKPT}/lifs_stad_fixed.csv")
cols = [c for c in df.columns if c not in ("filename", "vendor", "stage")]
V, st, fn = df.vendor.astype(str).values, df.stage.values, df.filename.values
X = df[cols].values.astype(float)

print("stage distribution by center, % (case-mix differs; label-free ComBat removes it along with the scanner effect):")
print((pd.crosstab(df.vendor, df.stage, normalize="index") * 100).round(0).to_string())

# label-free preparation: missing values -> same center's median; drop features constant within any center
gmed = np.nanmedian(X, 0)
for v in np.unique(V):
    m = V == v; Xm = X[m]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore"); med = np.nanmedian(Xm, 0)
    med = np.where(np.isnan(med), gmed, med)
    r, c = np.where(np.isnan(Xm)); Xm[r, c] = med[c]; X[m] = Xm
keep = np.all([X[V == v].std(0) > 1e-6 for v in np.unique(V)], axis=0)
print(f"\nfeatures kept for ComBat: {keep.sum()} of {len(cols)}")
Xc = np.asarray(pycombat_norm(X[:, keep].T, list(V))).T        # all 460 jointly, batch = scanner, no labels used
assert np.isfinite(Xc).all(), "ComBat produced non-finite values"

rf = lambda: make_pipeline(FunctionTransformer(np.nan_to_num),
                           RandomForestClassifier(n_estimators=600, min_samples_leaf=2, max_features="sqrt",
                                                  class_weight="balanced", n_jobs=-1, random_state=42))
lr = lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000))
rows, cases = [], pd.DataFrame({"filename": fn, "held": V, "stage": st})
for name, mk in [("rf", rf), ("logreg", lr)]:
    score = np.zeros(len(fn))
    for h in CENTERS:
        tr = V != h
        for t, thr in TASKS.items():
            p = mk().fit(Xc[tr], st[tr] >= thr).predict_proba(Xc[~tr])[:, 1]
            score[~tr] += p
            rows.append(dict(model=name, held=h, task=t, combat=roc_auc_score(st[~tr] >= thr, p)))
    cases[f"{name}_combat"] = score
U = pd.DataFrame(rows)
E = pd.read_csv(f"{RES}/E_baselines.csv"); E = E[E.feat == "fixed7"]
U = U.merge(E.pivot_table(index=["model", "held", "task"], columns="variant", values="auc").reset_index(),
            on=["model", "held", "task"], how="left")
print("\nLOCO AUC, 7-phase radiomic features:\n" + U.round(3).to_string(index=False))
cmp = [c for c in ["raw", "center-z", "center-rank", "combat"] if c in U.columns]
print("\nmean over 6 pairs:\n" + U.groupby("model")[cmp].mean().round(3).to_string())
print("\nmean over the 5 reliable pairs (B2 substantial excluded):\n" +
      U[~((U.held == "B2") & (U.task == "sub"))].groupby("model")[cmp].mean().round(3).to_string())
U.to_csv(f"{RES}/U_combat.csv", index=False); cases.to_csv(f"{RES}/U_combat_cases.csv", index=False)

# does ComBat remove scanner identity? (same 5-fold test as cell D; raw 7-phase features gave AUC 0.995)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score
cl = np.unique(V)
Pv = cross_val_predict(RandomForestClassifier(n_estimators=600, class_weight="balanced", n_jobs=-1, random_state=42),
                       Xc, V, cv=StratifiedKFold(5, shuffle=True, random_state=42), method="predict_proba")
print(f"\nscanner decodability after ComBat: AUC {roc_auc_score(V, Pv, multi_class='ovr', labels=cl):.3f}, "
      f"accuracy {accuracy_score(V, cl[Pv.argmax(1)]):.3f}   (raw features: 0.995 / 0.957)")

**Step 18.** CELL V: primary endpoint, family 1 (Table 3).

In [ ]:
# ===================== CELL V (CPU, ~3 min): PRIMARY endpoint for the existing models =====================
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, os, warnings
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
warnings.filterwarnings("ignore")
CKPT = "/content/drive/MyDrive/CARE2026_ckpts"; RES = f"{CKPT}/paper_results"
CENTERS = ["A", "B1", "B2"]

dp = pd.read_csv(f"{CKPT}/lifs_loco_deep/loco_deep_preds.csv")
D = dp.groupby(["filename", "held", "stage"], as_index=False)[["d1", "d4"]].mean()
D["deep"] = (1 - D.d1) + D.d4
BC = pd.read_csv(f"{RES}/B_loco_cases.csv")
for feat in ["fixed7", "ged4"]:
    for var in ["raw", "center-z", "center-rank"]:
        g = BC[(BC.feat == feat) & (BC.variant == var)]
        if len(g) == 0: continue
        D = D.merge(pd.DataFrame({"filename": g.filename.values, f"rf_{feat}_{var}": (g.rf_sub + g.rf_cir).values}),
                    on="filename", how="left")
        D[f"fusion_{feat}_{var}"] = 0.5 * D.deep + 0.5 * D[f"rf_{feat}_{var}"]
emb = f"{RES}/fm_dinov2_embeddings.npz"
if os.path.exists(emb):
    e = np.load(emb, allow_pickle=True)
    Xe, Ve, se, fe = e["emb"], e["vendor"].astype(str), e["stage"].astype(int), e["filename"].astype(str)
    lr = lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000))
    s = np.zeros(len(fe))
    for h in CENTERS:
        tr = Ve != h
        for thr in (2, 4): s[~tr] += lr().fit(Xe[tr], se[tr] >= thr).predict_proba(Xe[~tr])[:, 1]
    D = D.merge(pd.DataFrame({"filename": fe, "dinov2": s}), on="filename", how="left")
if os.path.exists(f"{RES}/U_combat_cases.csv"):
    D = D.merge(pd.read_csv(f"{RES}/U_combat_cases.csv")[["filename", "rf_combat", "logreg_combat"]], on="filename", how="left")

methods = [c for c in D.columns if c not in ("filename", "held", "stage", "d1", "d4")]
bad = [c for c in methods if D[c].isna().any()]
if bad: print("skipped (missing predictions):", bad)
methods = [c for c in methods if c not in bad]
print(f"{len(D)} cases | models: {methods}")

A = {h: (D[D.held == h].stage.values, {m: D[D.held == h][m].values for m in methods}) for h in CENTERS}
def pooled_all(idx):
    c = {m: 0.0 for m in methods}; n = 0
    for h in CENTERS:
        st = A[h][0][idx[h]]; comp = st[:, None] > st[None, :]; n += comp.sum()
        for m in methods:
            x = A[h][1][m][idx[h]]; dx = (x[:, None] - x[None, :])[comp]
            c[m] += (dx > 0).sum() + 0.5 * (dx == 0).sum()
    return np.array([c[m] / n for m in methods])
full = {h: np.arange(len(A[h][0])) for h in CENTERS}
point = pooled_all(full)
rng = np.random.default_rng(0); B = 2000
boots = np.array([pooled_all({h: rng.integers(0, len(A[h][0]), len(A[h][0])) for h in CENTERS}) for _ in range(B)])
lo, hi = np.percentile(boots, [2.5, 97.5], axis=0)
T = pd.DataFrame({"model": methods, "C_index": point, "ci": [f"{a:.3f}-{b:.3f}" for a, b in zip(lo, hi)]})
per = {h: [] for h in CENTERS}
for h in CENTERS:
    one = {k: (np.arange(len(A[k][0])) if k == h else np.array([], int)) for k in CENTERS}
    per[h] = pooled_all(one)
for h in CENTERS: T[f"C_{h}"] = per[h]
print("\npooled within-center C-index (held-out centers), 95% CI:\n" + T.sort_values("C_index", ascending=False).round(3).to_string(index=False))

j = {m: i for i, m in enumerate(methods)}
fam = [("fusion vs deep", "fusion_fixed7_raw"), ("radiomic RF vs deep", "rf_fixed7_raw"), ("frozen DINOv2 vs deep", "dinov2")]
fam = [(lab, m) for lab, m in fam if m in j]
P = []
for lab, m in fam:
    d = boots[:, j[m]] - boots[:, j["deep"]]
    p = max(1 / B, min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))
    a, b = np.percentile(d, [2.5, 97.5])
    P.append(dict(comparison=lab, delta=point[j[m]] - point[j["deep"]], ci=f"{a:+.3f},{b:+.3f}", p=p))
P = pd.DataFrame(P).sort_values("p").reset_index(drop=True)
k, stop, dec = len(P), False, []
for i, r in P.iterrows():
    ok = (not stop) and r.p <= 0.05 / (k - i); stop = stop or not ok; dec.append("significant" if ok else "n.s.")
P["holm_0.05"] = dec
print("\nPRIMARY comparisons (family 1, Holm-corrected):\n" + P.round(4).to_string(index=False))
T.to_csv(f"{RES}/V_concordance.csv", index=False); P.to_csv(f"{RES}/V_primary.csv", index=False)

**Step 19.** CELL T (GPU session): three series crops on the same grid, for the multiphase input. Run it in the same Colab session as CELL R2, which reads these crops from the Colab disk. Its last line names RUN = ['multiphase'] from our first session; here CELL R2 runs all three interventions.

In [ ]:
# ===================== CELL T (GPU session, CPU work ~25 min): 3-phase crops on the exact v2 GED4 grid =====================
!pip install -q SimpleITK 2>&1 | tail -1
from google.colab import drive
drive.mount('/content/drive')
import os, glob, time, shutil, zipfile, numpy as np, SimpleITK as sitk
from concurrent.futures import ThreadPoolExecutor        # threads, not processes: SimpleITK releases the GIL, no fork risk
CKPT  = "/content/drive/MyDrive/CARE2026_ckpts"
DATA  = "/content/Liver-Train-data/Liver-Train"
MASK_DIRS = [f"{CKPT}/lifs_train_masks_extra", f"{CKPT}/lifs_train_masks_st2_345"]   # the masks the v2 crops used
V2    = f"{CKPT}/lifs_ged4_hires_train_v2"
LOCAL, DRIVE_OUT = "/content/mp_crops", f"{CKPT}/lifs_mp_crops"
PHASES = ["GED4", "T2", "DWI_800"]                 # fixed in advance: three distinct MR contrasts
HI, PAD_FRAC = (24, 256, 256), 0.10
os.makedirs(LOCAL, exist_ok=True)

if len(glob.glob(f"{DATA}/Vendor_*/*/GED4.nii.gz")) < 460:
    t0 = time.time()
    shutil.copy("/content/drive/MyDrive/Liver-Train.zip", "/content/Liver-Train.zip"); print(f"zip copied ({time.time()-t0:.0f}s)")
    with zipfile.ZipFile("/content/Liver-Train.zip") as z:          # only the three phases we need: faster
        z.extractall("/content/Liver-Train-data", members=[n for n in z.namelist() if "__MACOSX" not in n
                                                           and n.endswith(tuple(f"/{p}.nii.gz" for p in PHASES))])
    os.remove("/content/Liver-Train.zip"); print(f"extracted ({time.time()-t0:.0f}s)")
BACKUP_TO_DRIVE = False     # True costs ~5 min; only needed if this session might disconnect before R2 finishes

mask_map = {}
for md in MASK_DIRS:
    for f in glob.glob(f"{md}/*.nii.gz"): mask_map[os.path.basename(f)[:-7]] = f
cases = sorted(os.path.basename(f)[:-4] for f in glob.glob(f"{V2}/*.npz"))
print(f"cases {len(cases)} | masks {len(mask_map)} | raw GED4 {len(glob.glob(f'{DATA}/Vendor_*/*/GED4.nii.gz'))}")

def span(lo, hi, n): p = int((hi - lo) * PAD_FRAC); return max(0, lo - p), min(n, hi + p + 1)
def crop_resize(raw, box):                         # identical to cell 127
    (z0, z1), (y0, y1), (x0, x1) = box
    crop = raw[z0:z1, y0:y1, x0:x1]; ci = sitk.GetImageFromArray(crop)
    ci_r = sitk.Resample(ci, [HI[2], HI[1], HI[0]], sitk.Transform(), sitk.sitkLinear, ci.GetOrigin(),
                         [crop.shape[2]/HI[2], crop.shape[1]/HI[1], crop.shape[0]/HI[0]], ci.GetDirection(), 0, sitk.sitkFloat32)
    out = sitk.GetArrayFromImage(ci_r)
    return (out - out.mean()) / (out.std() + 1e-6), crop

def build(case):
    out_p = f"{LOCAL}/{case}.npy"
    if os.path.exists(out_p): return case, "cached", None
    try:
        img = sitk.ReadImage(glob.glob(f"{DATA}/Vendor_*/{case}/GED4.nii.gz")[0], sitk.sitkFloat32)
        mask = sitk.Resample(sitk.ReadImage(mask_map[case]), img, sitk.Transform(), sitk.sitkNearestNeighbor, 0)
        raw = sitk.GetArrayFromImage(img); zz, yy, xx = np.where(sitk.GetArrayFromImage(mask) > 0)
        box = (span(zz.min(), zz.max(), raw.shape[0]), span(yy.min(), yy.max(), raw.shape[1]), span(xx.min(), xx.max(), raw.shape[2]))
        chans, cover = [], []
        for ph in PHASES:
            if ph == "GED4":
                arr = raw
            else:
                pp = glob.glob(f"{DATA}/Vendor_*/{case}/{ph}.nii.gz")
                try:
                    src = sitk.ReadImage(pp[0], sitk.sitkFloat32)          # other phase -> GED4 grid, in physical space
                    arr = sitk.GetArrayFromImage(sitk.Resample(src, img, sitk.Transform(), sitk.sitkLinear, 0.0, sitk.sitkFloat32))
                except Exception:
                    chans.append(np.zeros(HI, np.float32)); cover.append(0.0); continue
            o, crop = crop_resize(arr, box)
            chans.append(o.astype(np.float32)); cover.append(float((crop != 0).mean()))
        np.save(out_p, np.stack(chans, 1).astype(np.float16))           # (24, 3, 256, 256)
        return case, "ok", cover
    except Exception as e:
        return case, "FAIL " + str(e)[:80], None

sitk.ProcessObject_SetGlobalDefaultNumberOfThreads(1)    # one ITK thread per case; cases run in parallel threads
t0 = time.time(); res = []
with ThreadPoolExecutor(max_workers=min(8, os.cpu_count() or 2)) as ex:
    for i, r in enumerate(ex.map(build, cases), 1):
        res.append(r)
        if i % 50 == 0: print(f"  {i}/{len(cases)} ({time.time()-t0:.0f}s)")
sitk.ProcessObject_SetGlobalDefaultNumberOfThreads(os.cpu_count() or 2)
fails = [r for r in res if r[1].startswith("FAIL")]
cov = np.array([r[2] for r in res if r[2] is not None])
print(f"\nbuilt {sum(r[1] == 'ok' for r in res)} | cached {sum(r[1] == 'cached' for r in res)} | FAILED {len(fails)} {fails[:3]}")
if len(cov):
    print("median fraction of the crop covered by each phase:", dict(zip(PHASES, np.median(cov, 0).round(3))))
    print("cases where a phase is missing or unreadable (filled with zeros):", dict(zip(PHASES, (cov == 0).sum(0).tolist())))

# check 1: the GED4 channel must reproduce the stored v2 crops (same code path, so r should be ~1)
rs = [np.corrcoef(np.load(f"{LOCAL}/{c}.npy")[:, 0].astype(np.float32).ravel(), np.load(f"{V2}/{c}.npz")["ged4"].ravel())[0, 1]
      for c in np.random.default_rng(0).choice(cases, 25, replace=False)]
print(f"GED4 channel vs stored v2 crops: min r = {min(rs):.5f}  (must be > 0.999)")
assert min(rs) > 0.999, "GED4 channel does not match the v2 crops: STOP, do not train"
# check 2: all cases present
done_ = [c for c in cases if os.path.exists(f"{LOCAL}/{c}.npy")]
assert len(done_) == 460, f"only {len(done_)} of 460 crops built"
# check 3: the extra phases line up with GED4. Contrasts differ between phases, so compare EDGE maps (organ boundaries),
#          and compare against a deliberately shifted copy: aligned must score clearly higher than shifted.
def edges(v):
    gy, gx = np.gradient(v, axis=(1, 2)); return np.hypot(gy, gx).ravel()
al = []
for c in np.random.default_rng(1).choice(cases, 25, replace=False):
    x = np.load(f"{LOCAL}/{c}.npy").astype(np.float32); e0 = edges(x[:, 0]); row = []
    for k in (1, 2):
        if x[:, k].std() == 0: row += [np.nan, np.nan]; continue
        row += [np.corrcoef(e0, edges(x[:, k]))[0, 1], np.corrcoef(e0, edges(np.roll(x[:, k], 32, axis=2)))[0, 1]]
    al.append(row)
med = np.nanmedian(al, 0)
print(f"edge alignment with GED4, aligned vs shifted by 32 px:  T2 {med[0]:.3f} vs {med[1]:.3f}  |  DWI_800 {med[2]:.3f} vs {med[3]:.3f}")
print("  -> aligned must be clearly higher than shifted for both; if not, STOP and check the crops")
if BACKUP_TO_DRIVE:
    os.makedirs(DRIVE_OUT, exist_ok=True); t0 = time.time()
    for c in done_:
        if not os.path.exists(f"{DRIVE_OUT}/{c}.npy"): shutil.copy(f"{LOCAL}/{c}.npy", f"{DRIVE_OUT}/{c}.npy")
    print(f"backed up to Drive ({time.time()-t0:.0f}s).")
print("Crops ready. Now run CELL R2 in THIS session with RUN = ['multiphase'].")

**Step 20.** CELL R2 (GPU, about 1.5 hours per intervention, resumable): the three interventions. We ran multiphase in one Colab session and the other two together in a second.

In [ ]:
# ===================== CELL R2 (GPU, ~4.5 h unattended, resumable): the three pre-declared interventions =====================
!pip install -q timm 2>&1 | tail -1
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, glob, os, time, random, shutil
import torch, torch.nn as nn, torch.nn.functional as F, timm
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from scipy.ndimage import gaussian_filter
from sklearn.metrics import roc_auc_score
torch.backends.cudnn.benchmark = True
CKPT  = "/content/drive/MyDrive/CARE2026_ckpts"
HIRES = f"{CKPT}/lifs_ged4_hires_train_v2"
MP_LOCAL, MP_DRIVE = "/content/mp_crops", f"{CKPT}/lifs_mp_crops"
OUT   = f"{CKPT}/lifs_dg"; os.makedirs(OUT, exist_ok=True)
RUN = ["multiphase", "intensity_aug", "adversarial"]   # we ran multiphase in one Colab session and the other two in a second; each is resumable
METHODS = [m for m in ["multiphase", "intensity_aug", "adversarial"] if m in RUN]
SEEDS, CENTERS, ADV_WEIGHT = [42, 123, 7], ["A", "B1", "B2"], 0.1
print("this session runs:", METHODS)

files = sorted(glob.glob(f"{HIRES}/*.npz"))
CASE = {f: os.path.basename(f)[:-4] for f in files}
VEND, STAGE = {}, {}
for f in files:
    d = np.load(f); VEND[f] = str(d["vendor"]); STAGE[f] = int(d["stage"])

if "multiphase" in METHODS:
    if len(glob.glob(f"{MP_LOCAL}/*.npy")) < 460 and len(glob.glob(f"{MP_DRIVE}/*.npy")) >= 460:
        os.makedirs(MP_LOCAL, exist_ok=True); t0 = time.time()
        for p in glob.glob(f"{MP_DRIVE}/*.npy"):
            q = f"{MP_LOCAL}/{os.path.basename(p)}"
            if not os.path.exists(q): shutil.copy(p, q)
        print(f"multi-phase crops restored from Drive ({time.time()-t0:.0f}s)")
    assert len(glob.glob(f"{MP_LOCAL}/*.npy")) >= 460, "multi-phase crops not found: run CELL T in THIS session first"

def load_x(f, method):
    if method == "multiphase": return np.load(f"{MP_LOCAL}/{CASE[f]}.npy").astype(np.float32)   # (24, 3, 256, 256)
    return np.load(f)["ged4"].astype(np.float32)                                               # (24, 256, 256)

def scanner_style_aug(x):
    """Label-free MR appearance changes on a z-scored (24,H,W) crop; output is re-z-scored like the inputs."""
    r = np.random.default_rng(random.getrandbits(32))
    lo, hi = np.percentile(x, [0.5, 99.5])
    u = np.clip((x - lo) / (hi - lo + 1e-6), 0, None)
    if random.random() < 0.8:
        u = u ** np.exp(r.uniform(np.log(0.7), np.log(1.5)))
    if random.random() < 0.5:
        H, W = u.shape[1:]
        yy, xx = np.meshgrid(np.linspace(-1, 1, H), np.linspace(-1, 1, W), indexing="ij")
        c = r.uniform(-0.3, 0.3, 3)
        u = u * np.exp(c[0]*yy + c[1]*xx + c[2]*xx*yy)[None]
    if random.random() < 0.5:
        u = gaussian_filter(u, sigma=(0, r.uniform(0.3, 1.2), r.uniform(0.3, 1.2)))
    if random.random() < 0.5:
        u = u + r.normal(0, r.uniform(0.01, 0.05), u.shape)
    return ((u - u.mean()) / (u.std() + 1e-6)).astype(np.float32)

class DS(Dataset):
    def __init__(s, fl, vmap, method): s.fl, s.vmap, s.method = fl, vmap, method
    def __len__(s): return len(s.fl)
    def __getitem__(s, i):
        f = s.fl[i]; x = load_x(f, s.method)
        if s.method == "intensity_aug": x = scanner_style_aug(x)
        if random.random() < .5: x = x[..., ::-1]
        if random.random() < .5: x = x[..., ::-1, :]
        return torch.from_numpy(x.copy()), STAGE[f] - 1, s.vmap[VEND[f]]

class SliceAttnNet(nn.Module):
    def __init__(s, n=4, in_ch=1, pretrained=True):
        super().__init__()
        s.enc = timm.create_model("convnext_tiny", pretrained=pretrained, in_chans=in_ch, num_classes=0, global_pool="avg")
        f = s.enc.num_features
        s.attn = nn.Sequential(nn.Linear(f, 128), nn.Tanh(), nn.Linear(128, 1))
        s.head = nn.Sequential(nn.Dropout(0.5), nn.Linear(f, n))
    def embed(s, x):
        if x.dim() == 4: x = x.unsqueeze(2)                     # (B,S,H,W) -> (B,S,1,H,W): same as the baseline
        B, S, C, H, W = x.shape
        fe = s.enc(x.reshape(B*S, C, H, W)).reshape(B, S, -1)
        a = torch.softmax(s.attn(fe).squeeze(-1), 1)
        return (fe*a.unsqueeze(-1)).sum(1)
    def forward(s, x): return s.head(s.embed(x))

class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, a): ctx.a = a; return x.view_as(x)
    @staticmethod
    def backward(ctx, g): return -ctx.a * g, None

IN_CH = lambda method: 3 if method == "multiphase" else 1

def train(tr_files, seed, method, epochs=40, max_steps=None):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    cnt = Counter(STAGE[f]-1 for f in tr_files); tot = sum(cnt.values())
    cw = torch.tensor([tot/(4*max(cnt[c], 1)) for c in range(4)], dtype=torch.float32).cuda()
    vend = sorted({VEND[f] for f in tr_files}); vmap = {v: i for i, v in enumerate(vend)}
    dl = DataLoader(DS(tr_files, vmap, method), batch_size=4, shuffle=True, num_workers=4, drop_last=True)
    m = SliceAttnNet(in_ch=IN_CH(method)).cuda(); params = list(m.parameters())
    if method == "adversarial":
        vh = nn.Sequential(nn.Linear(m.enc.num_features, 128), nn.ReLU(), nn.Linear(128, len(vend))).cuda()
        params += list(vh.parameters())
    opt = torch.optim.AdamW(params, lr=1e-4, weight_decay=1e-5)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs); sc = torch.amp.GradScaler('cuda'); m.train()
    n_steps, step = epochs * len(dl), 0
    for ep in range(epochs):
        t0 = time.time()
        for x, y, v in dl:
            x, y, v = x.cuda(), y.cuda(), v.cuda(); opt.zero_grad()
            with torch.amp.autocast('cuda'):
                z = m.embed(x)
                ce = F.cross_entropy(m.head(z), y, weight=cw, reduction="none")
                loss = torch.topk(ce, max(1, int(0.8*ce.numel()))).values.mean()         # FairCVaR, as deployed
                if method == "adversarial":
                    a = float(2 / (1 + np.exp(-10 * step / n_steps)) - 1)              # standard DANN ramp 0 -> 1
                    loss = loss + ADV_WEIGHT * F.cross_entropy(vh(GradReverse.apply(z, a)), v)
            sc.scale(loss).backward(); sc.step(opt); sc.update(); step += 1
            if max_steps and step >= max_steps: return m.eval()
        sch.step()
        if ep % 10 == 0 or ep == epochs-1: print(f"    {method} seed{seed} ep{ep:02d} loss {loss.item():.3f} ({time.time()-t0:.0f}s)")
    return m.eval()

@torch.no_grad()
def predict(m, fl, method):
    out = []
    for f in fl:
        x = load_x(f, method)
        P = [F.softmax(m(torch.from_numpy(v.copy())[None].cuda().float()), 1)[0].cpu().numpy()
             for v in (x, x[..., ::-1], x[..., ::-1, :], x[..., ::-1, ::-1])]
        out.append(np.mean(P, 0))
    return np.array(out)

@torch.no_grad()
def embed_all(m, fl, method):
    return np.array([m.embed(torch.from_numpy(load_x(f, method))[None].cuda()).float().cpu().numpy()[0] for f in fl])

# smoke test: one optimisation step + prediction + embedding per method (fails within minutes if anything is wrong)
for method in METHODS:
    _m = train(files[:16], 0, method, epochs=1, max_steps=1)
    _p = predict(_m, files[:2], method); _e = embed_all(_m, files[:2], method)
    assert _p.shape == (2, 4) and np.isfinite(_p).all() and _e.shape[0] == 2, f"smoke test failed for {method}"
    del _m; torch.cuda.empty_cache()
print("smoke test passed for:", METHODS, "\n")

for method in METHODS:
    rows = []                                                   # one CSV per method, so parallel sessions never collide
    for s in SEEDS:
        for held in CENTERS:
            te = [f for f in files if VEND[f] == held]; tr = [f for f in files if VEND[f] != held]
            tag = f"{method}_{held}_seed{s}"; pp, mp = f"{OUT}/preds_{tag}.npy", f"{OUT}/model_{tag}.pth"
            if os.path.exists(pp):
                P = np.load(pp); print(f"{tag}: cached")
            else:
                if os.path.exists(mp):
                    m = SliceAttnNet(in_ch=IN_CH(method), pretrained=False).cuda(); m.load_state_dict(torch.load(mp)); m.eval()
                else:
                    print(f"=== {tag}: training on {len(tr)} ==="); m = train(tr, s, method); torch.save(m.state_dict(), mp)
                P = predict(m, te, method); np.save(pp, P)
                if held == "B2" and s == 42:
                    np.savez(f"{OUT}/emb_{method}_B2_seed42.npz", filename=np.array([CASE[f] for f in files]),
                             vendor=np.array([VEND[f] for f in files]), emb=embed_all(m, files, method))
                del m; torch.cuda.empty_cache()
            st = np.array([STAGE[f] for f in te])
            print(f"  {tag}: sub {roc_auc_score(st >= 2, 1 - P[:, 0]):.3f}  cir {roc_auc_score(st >= 4, P[:, 3]):.3f}")
            rows += [dict(filename=CASE[f], held=held, seed=s, method=method, stage=STAGE[f],
                          d1=p[0], d2=p[1], d3=p[2], d4=p[3]) for f, p in zip(te, P)]
            pd.DataFrame(rows).to_csv(f"{OUT}/dg_preds_{method}.csv", index=False)      # saved after every run
print("\nall runs finished for:", METHODS)

**Step 21.** CELL S2: family 2 (Table 3 and Sec. S1).

In [ ]:
# ===================== CELL S2 (CPU, any time, ~2 min): evaluate the interventions (family 2) =====================
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, accuracy_score
CKPT = "/content/drive/MyDrive/CARE2026_ckpts"; RES = f"{CKPT}/paper_results"; OUT = f"{CKPT}/lifs_dg"
CENTERS, SEEDS = ["A", "B1", "B2"], [42, 123, 7]
METHODS = ["multiphase", "intensity_aug", "adversarial"]
RELIABLE = [("A", "sub"), ("A", "cir"), ("B1", "sub"), ("B1", "cir"), ("B2", "cir")]

base = pd.read_csv(f"{CKPT}/lifs_loco_deep/loco_deep_preds.csv").assign(method="baseline")
import glob as _glob
_parts = [pd.read_csv(p) for p in sorted(_glob.glob(f"{OUT}/dg_preds_*.csv"))]
dg = pd.concat(_parts, ignore_index=True) if _parts else pd.DataFrame(columns=list(base.columns) + ["method"])
E = pd.concat([base, dg], ignore_index=True); E["p_sub"] = 1 - E.d1.astype(float); E["p_cir"] = E.d4.astype(float)
def complete_seeds(method):
    return [s for s in SEEDS if all(((E.method == method) & (E.seed == s) & (E.held == h)).any() for h in CENTERS)]
def ens(method, seeds):
    return E[(E.method == method) & E.seed.isin(seeds)].groupby(["filename", "held", "stage"], as_index=False)[["p_sub", "p_cir"]].mean()
def pooled_c(A, key, idx):
    c = n = 0
    for h in CENTERS:
        st, x = A[h]["st"][idx[h]], A[h][key][idx[h]]
        comp = st[:, None] > st[None, :]; dx = (x[:, None] - x[None, :])[comp]
        c += (dx > 0).sum() + 0.5 * (dx == 0).sum(); n += comp.sum()
    return c / n
def mean_auc(A, sfx, idx):
    v = []
    for h, t in RELIABLE:
        ii = idx[h]; y = (A[h]["st"][ii] >= (2 if t == "sub" else 4)).astype(int)
        if y.min() == y.max(): return np.nan
        v.append(roc_auc_score(y, A[h][f"p{t[0]}_{sfx}"][ii]))
    return np.mean(v)

rng = np.random.default_rng(0); B = 2000; rows = []
for method in METHODS:
    seeds = complete_seeds(method)
    if not seeds: print(f"{method}: nothing finished yet"); continue
    m = ens(method, seeds).merge(ens("baseline", seeds), on=["filename", "held", "stage"], suffixes=("_m", "_b"))
    A = {}
    for h in CENTERS:
        g = m[m.held == h]
        A[h] = dict(st=g.stage.values.astype(int), sm=(g.p_sub_m + g.p_cir_m).values, sb=(g.p_sub_b + g.p_cir_b).values,
                    ps_m=g.p_sub_m.values, pc_m=g.p_cir_m.values, ps_b=g.p_sub_b.values, pc_b=g.p_cir_b.values)
    full = {h: np.arange(len(A[h]["st"])) for h in CENTERS}
    dC, dA = [], []
    for _ in range(B):
        idx = {h: rng.integers(0, len(A[h]["st"]), len(A[h]["st"])) for h in CENTERS}
        dC.append(pooled_c(A, "sm", idx) - pooled_c(A, "sb", idx)); dA.append(mean_auc(A, "m", idx) - mean_auc(A, "b", idx))
    dC, dA = np.array(dC), np.array(dA)
    seedC = [pooled_c({h: dict(st=g.stage.values.astype(int), s=(g.p_sub + g.p_cir).values)
                       for h in CENTERS for g in [E[(E.method == method) & (E.seed == s) & (E.held == h)]]}, "s",
                      {h: np.arange(((E.method == method) & (E.seed == s) & (E.held == h)).sum()) for h in CENTERS}) for s in seeds]
    rows.append(dict(method=method, seeds=str(seeds), final=len(seeds) == 3,
                     C_method=pooled_c(A, "sm", full), C_baseline=pooled_c(A, "sb", full),
                     dC=pooled_c(A, "sm", full) - pooled_c(A, "sb", full),
                     dC_ci="{:+.3f},{:+.3f}".format(*np.percentile(dC, [2.5, 97.5])),
                     p=max(1 / B, min(1.0, 2 * min((dC <= 0).mean(), (dC >= 0).mean()))),
                     dAUC_5pairs=mean_auc(A, "m", full) - mean_auc(A, "b", full),
                     dAUC_ci="{:+.3f},{:+.3f}".format(*np.nanpercentile(dA, [2.5, 97.5])),
                     C_single_seed=f"{np.mean(seedC):.3f}+/-{np.std(seedC):.3f}"))
R = pd.DataFrame(rows)
if len(R):
    fin = R[R.final].sort_values("p"); k, stop, dec = len(fin), False, {}
    for i, (ix, r) in enumerate(fin.iterrows()):
        ok = (not stop) and r.p <= 0.05 / (k - i); stop = stop or not ok; dec[ix] = "significant" if ok else "n.s."
    R["holm_0.05"] = [dec.get(i, "preliminary (seeds incomplete)") for i in R.index]
    print("FAMILY 2: interventions vs deep baseline (seed-matched; primary = pooled within-center C-index)\n" +
          R.round(4).to_string(index=False))
    R.to_csv(f"{RES}/S2_interventions.csv", index=False)

def vendor_dec(path):
    e = np.load(path, allow_pickle=True); V = e["vendor"].astype(str); cl = np.unique(V)
    P = cross_val_predict(RandomForestClassifier(n_estimators=600, class_weight="balanced", n_jobs=-1, random_state=42),
                          e["emb"], V, cv=StratifiedKFold(5, shuffle=True, random_state=42), method="predict_proba")
    return roc_auc_score(V, P, multi_class="ovr", labels=cl), accuracy_score(V, cl[P.argmax(1)])
print("\nscanner decodability of the B2-held-out, seed-42 embedding (lower = less scanner information):")
for name, p in [("baseline", f"{RES}/deep_embeddings.npz")] + [(mm, f"{OUT}/emb_{mm}_B2_seed42.npz") for mm in METHODS]:
    if os.path.exists(p): a, acc = vendor_dec(p); print(f"  {name:14s} AUC {a:.3f}  accuracy {acc:.3f}")
    else: print(f"  {name:14s} not available yet")

**Step 22.** CELL W: nested fusion that uses no labels from the held out scanner (Table 3).

In [ ]:
# ===================== CELL W (CPU, ~5 min): nested, target-label-free fusion of deep + harmonized radiomics =====================
# Design, fixed before running. It was motivated by the center-level pattern seen in cell V, so report it as a
# post hoc analysis whose component selection is nested:
#  * Radiomic stream: for each held-out center, ONE pipeline is chosen among {RF, logistic} x {raw, center-z, center-rank, ComBat}
#    by inner leave-one-center-out on the TWO training centers only (mean within-center C-index over both directions).
#    The held-out center's labels are never used for any choice.
#  * Fusion: average of the deep and radiomic risk scores, each standardized within the held-out center (label-free).
#  * Test: nested fusion vs deep on the primary endpoint (pooled within-center C-index), paired bootstrap.
!pip install -q inmoose 2>&1 | tail -1
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, os, warnings
from scipy.stats import rankdata
from inmoose.pycombat import pycombat_norm
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import make_pipeline
warnings.filterwarnings("ignore")
CKPT = "/content/drive/MyDrive/CARE2026_ckpts"; RES = f"{CKPT}/paper_results"
CENTERS = ["A", "B1", "B2"]

df = pd.read_csv(f"{CKPT}/lifs_stad_fixed.csv")
cols = [c for c in df.columns if c not in ("filename", "vendor", "stage")]
V, st, fn = df.vendor.astype(str).values, df.stage.values.astype(int), df.filename.values
X0 = df[cols].values.astype(float)

def center_z(X):
    Z = np.full_like(X, np.nan)
    for v in CENTERS:
        m = V == v; sd = np.nanstd(X[m], 0); sd[~np.isfinite(sd) | (sd < 1e-8)] = 1.0
        Z[m] = (X[m] - np.nanmean(X[m], 0)) / sd
    return Z
def center_rank(X):
    R = np.full_like(X, np.nan)
    for v in CENTERS:
        idx = np.where(V == v)[0]
        for j in range(X.shape[1]):
            col = X[idx, j]; ok = np.isfinite(col)
            if ok.sum() > 1:
                r = np.full(len(col), np.nan); r[ok] = (rankdata(col[ok]) - 0.5) / ok.sum(); R[idx, j] = r
    return R
def combat(X):                                       # identical preparation to cell U
    X = X.copy(); gmed = np.nanmedian(X, 0)
    for v in CENTERS:
        m = V == v; Xm = X[m]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore"); med = np.nanmedian(Xm, 0)
        med = np.where(np.isnan(med), gmed, med); r, c = np.where(np.isnan(Xm)); Xm[r, c] = med[c]; X[m] = Xm
    keep = np.all([X[V == v].std(0) > 1e-6 for v in CENTERS], axis=0)
    return np.asarray(pycombat_norm(X[:, keep].T, list(V))).T
FEAT = {"raw": X0, "center-z": center_z(X0), "center-rank": center_rank(X0), "combat": combat(X0)}
MODELS = {
    "rf": lambda: make_pipeline(SimpleImputer(strategy="median"), FunctionTransformer(np.nan_to_num),
                                RandomForestClassifier(n_estimators=600, min_samples_leaf=2, max_features="sqrt",
                                                       class_weight="balanced", n_jobs=-1, random_state=42)),
    "logreg": lambda: make_pipeline(SimpleImputer(strategy="median"), FunctionTransformer(np.nan_to_num), StandardScaler(),
                                    LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000))}
def risk(model, Xtr, s_tr, Xte):                    # risk score = P(stage>=2) + P(stage>=4), as everywhere else
    return sum(MODELS[model]().fit(Xtr, s_tr >= thr).predict_proba(Xte)[:, 1] for thr in (2, 4))
def cindex(stage, score):
    comp = stage[:, None] > stage[None, :]; dx = (score[:, None] - score[None, :])[comp]
    return ((dx > 0).sum() + 0.5 * (dx == 0).sum()) / comp.sum()

# ---- nested selection of the radiomic pipeline, separately for each held-out center ----
radio = np.full(len(fn), np.nan); log = []
for h in CENTERS:
    t1, t2 = [c for c in CENTERS if c != h]
    inner = {}
    for model in MODELS:
        for var, X in FEAT.items():
            inner[(model, var)] = np.mean([cindex(st[V == b], risk(model, X[V == a], st[V == a], X[V == b]))
                                           for a, b in ((t1, t2), (t2, t1))])
    best = max(inner, key=inner.get)
    tr, te = V != h, V == h
    radio[te] = risk(best[0], FEAT[best[1]][tr], st[tr], FEAT[best[1]][te])
    top = sorted(inner.items(), key=lambda kv: -kv[1])[:3]
    print(f"held-out {h}: chosen {best[0]} + {best[1]} (inner C {inner[best]:.3f}) | top-3 inner: "
          + ", ".join(f"{m}+{v} {c:.3f}" for (m, v), c in top))
    log.append(dict(held=h, chosen=f"{best[0]}+{best[1]}", inner_C=inner[best]))

# ---- fuse with the deep model and evaluate on the primary endpoint ----
dp = pd.read_csv(f"{CKPT}/lifs_loco_deep/loco_deep_preds.csv")
D = dp.groupby(["filename", "held", "stage"], as_index=False)[["d1", "d4"]].mean(); D["deep"] = 1 - D.d1 + D.d4
D = D.merge(pd.DataFrame({"filename": fn, "radio_nested": radio}), on="filename")
zc = lambda col: D.groupby("held")[col].transform(lambda s: (s - s.mean()) / (s.std() + 1e-12))
D["fusion_nested"] = 0.5 * zc("deep") + 0.5 * zc("radio_nested")
methods = ["deep", "radio_nested", "fusion_nested"]
if os.path.exists(f"{RES}/U_combat_cases.csv"):
    D = D.merge(pd.read_csv(f"{RES}/U_combat_cases.csv")[["filename", "logreg_combat"]], on="filename")
    D["fusion_posthoc"] = 0.5 * zc("deep") + 0.5 * zc("logreg_combat"); methods.append("fusion_posthoc")
assert D[methods].notna().all().all() and len(D) == 460, "missing predictions"

A = {h: (D[D.held == h].stage.values.astype(int), {m: D[D.held == h][m].values for m in methods}) for h in CENTERS}
def pooled(idx):
    c = {m: 0.0 for m in methods}; n = 0
    for h in CENTERS:
        s = A[h][0][idx[h]]; comp = s[:, None] > s[None, :]; n += comp.sum()
        for m in methods:
            x = A[h][1][m][idx[h]]; dx = (x[:, None] - x[None, :])[comp]; c[m] += (dx > 0).sum() + 0.5 * (dx == 0).sum()
    return np.array([c[m] / n for m in methods])
full = {h: np.arange(len(A[h][0])) for h in CENTERS}
point = pooled(full); rng = np.random.default_rng(0); B = 2000
boots = np.array([pooled({h: rng.integers(0, len(A[h][0]), len(A[h][0])) for h in CENTERS}) for _ in range(B)])
lo, hi = np.percentile(boots, [2.5, 97.5], axis=0)
per = {h: pooled({k: (np.arange(len(A[k][0])) if k == h else np.array([], int)) for k in CENTERS}) for h in CENTERS}
T = pd.DataFrame({"model": methods, "C_index": point, "ci": [f"{a:.3f}-{b:.3f}" for a, b in zip(lo, hi)],
                  **{f"C_{h}": per[h] for h in CENTERS}})
print("\npooled within-center C-index on held-out centers:\n" + T.round(3).to_string(index=False))
rows = []
for m, label in [("fusion_nested", "TEST: nested fusion vs deep"), ("radio_nested", "nested radiomics vs deep"),
                 ("fusion_posthoc", "post hoc fusion (ComBat+logreg) vs deep, NOT a valid test")]:
    if m not in methods: continue
    d = boots[:, methods.index(m)] - boots[:, 0]; a, b = np.percentile(d, [2.5, 97.5])
    p = max(1 / B, min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))
    rows.append(dict(comparison=label, delta=point[methods.index(m)] - point[0], ci=f"{a:+.3f},{b:+.3f}", p=p))
P = pd.DataFrame(rows); print("\n" + P.round(4).to_string(index=False))
verdict = "SIGNIFICANT improvement" if float(P.iloc[0].ci.split(",")[0]) > 0 else "no significant improvement"
print(f"\n=> nested fusion: {verdict} over the deep model")
T.to_csv(f"{RES}/W_nested_fusion.csv", index=False); P.to_csv(f"{RES}/W_nested_fusion_test.csv", index=False)
pd.DataFrame(log).to_csv(f"{RES}/W_nested_choices.csv", index=False)

**Step 23.** CELL 9 (GPU): in distribution cross validation with seeds 123 and 7.

In [ ]:
# CELL 9 (GPU, ~80 min unattended, resumable): in-distribution deep 5-fold CV, two more seeds
# October CELL J unchanged except the seed loop. Seed 42 files (preds_fold{k}.npy) are not touched.
!pip install -q timm 2>&1 | tail -1
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, glob, os, time, random
import torch, torch.nn as nn, torch.nn.functional as F, timm
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
torch.backends.cudnn.benchmark = True
CKPT  = "/content/drive/MyDrive/CARE2026_ckpts"
HIRES = f"{CKPT}/lifs_ged4_hires_train_v2"
RES   = f"{CKPT}/paper_results"
OUT   = f"{CKPT}/lifs_id_cv"
NEW_SEEDS = [123, 7]                      # the two LOSO seeds the in-distribution run lacked
files = sorted(glob.glob(f"{HIRES}/*.npz"))
VEND, STAGE = {}, {}
for f in files:
    d = np.load(f); VEND[f] = str(d["vendor"]); STAGE[f] = int(d["stage"])
class DS(Dataset):
    def __init__(s, fl): s.fl = fl
    def __len__(s): return len(s.fl)
    def __getitem__(s, i):
        x = np.load(s.fl[i])["ged4"].astype(np.float32); y = STAGE[s.fl[i]] - 1
        if random.random() < .5: x = x[:, :, ::-1]
        if random.random() < .5: x = x[:, ::-1, :]
        return torch.from_numpy(x.copy()), y
class SliceAttnNet(nn.Module):
    def __init__(s, n=4, pretrained=True):
        super().__init__()
        s.enc = timm.create_model("convnext_tiny", pretrained=pretrained, in_chans=1, num_classes=0, global_pool="avg")
        f = s.enc.num_features
        s.attn = nn.Sequential(nn.Linear(f, 128), nn.Tanh(), nn.Linear(128, 1))
        s.head = nn.Sequential(nn.Dropout(0.5), nn.Linear(f, n))
    def embed(s, x):
        B, S, H, W = x.shape
        fe = s.enc(x.reshape(B*S, 1, H, W)).reshape(B, S, -1)
        a = torch.softmax(s.attn(fe).squeeze(-1), 1)
        return (fe*a.unsqueeze(-1)).sum(1)
    def forward(s, x): return s.head(s.embed(x))
def train(tr_files, seed, epochs=40):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    cnt = Counter(STAGE[f]-1 for f in tr_files); tot = sum(cnt.values())
    cw = torch.tensor([tot/(4*max(cnt[c], 1)) for c in range(4)], dtype=torch.float32).cuda()
    dl = DataLoader(DS(tr_files), batch_size=4, shuffle=True, num_workers=2, drop_last=True)
    m = SliceAttnNet().cuda()
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4, weight_decay=1e-5)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs); sc = torch.amp.GradScaler('cuda'); m.train()
    for ep in range(epochs):
        t0 = time.time()
        for x, y in dl:
            x, y = x.cuda(), y.cuda(); opt.zero_grad()
            with torch.amp.autocast('cuda'):
                ce = F.cross_entropy(m(x), y, weight=cw, reduction="none")
                loss = torch.topk(ce, max(1, int(0.8*ce.numel()))).values.mean()
            sc.scale(loss).backward(); sc.step(opt); sc.update()
        sch.step()
        if ep % 10 == 0 or ep == epochs-1: print(f"    ep{ep:02d} ({time.time()-t0:.0f}s)")
    return m.eval()
@torch.no_grad()
def predict(m, fl):
    out = []
    for f in fl:
        x = np.load(f)["ged4"].astype(np.float32)
        P = [F.softmax(m(torch.from_numpy(v.copy())[None].cuda().float()), 1)[0].cpu().numpy()
             for v in (x, x[:, :, ::-1], x[:, ::-1, :], x[:, ::-1, ::-1])]
        out.append(np.mean(P, 0))
    return np.array(out)
strat = [f"{VEND[f]}_{STAGE[f]}" for f in files]
splits = list(StratifiedKFold(5, shuffle=True, random_state=42).split(files, strat))

old = pd.read_csv(f"{RES}/id_cv_deep_preds.csv")          # October, seed 42
for k, (tr, te) in enumerate(splits):
    assert set(os.path.basename(files[i])[:-4] for i in te) == set(old[old.fold == k].filename), f"fold {k} differs from October"
print("folds identical to October: OK")

for s in NEW_SEEDS:
    for k, (tr, te) in enumerate(splits):
        pp, mp = f"{OUT}/preds_fold{k}_seed{s}.npy", f"{OUT}/model_fold{k}_seed{s}.pth"
        if os.path.exists(pp):
            print(f"seed {s} fold {k}: cached"); continue
        if os.path.exists(mp):
            m = SliceAttnNet(pretrained=False).cuda(); m.load_state_dict(torch.load(mp)); m.eval()
        else:
            print(f"seed {s} fold {k}: training on {len(tr)}"); m = train([files[i] for i in tr], s); torch.save(m.state_dict(), mp)
        np.save(pp, predict(m, [files[i] for i in te])); del m; torch.cuda.empty_cache()

parts = [old.assign(seed=42)]
for s in NEW_SEEDS:
    for k, (tr, te) in enumerate(splits):
        P, tef = np.load(f"{OUT}/preds_fold{k}_seed{s}.npy"), [files[i] for i in te]
        parts.append(pd.DataFrame(dict(filename=[os.path.basename(f)[:-4] for f in tef], vendor=[VEND[f] for f in tef],
                                       stage=[STAGE[f] for f in tef], fold=k, d1=P[:, 0], d2=P[:, 1], d3=P[:, 2], d4=P[:, 3], seed=s)))
ID3 = pd.concat(parts, ignore_index=True); ID3.to_csv(f"{RES}/id_cv_deep_preds_3seeds.csv", index=False)
print(f"\nsaved {len(ID3)} rows -> id_cv_deep_preds_3seeds.csv")
for name, g in [("seed 42 only (as submitted)", old),
                ("3-seed mean", ID3.groupby(["filename", "vendor", "stage"], as_index=False)[["d1", "d4"]].mean())]:
    print(f"{name:28s} pooled AUC  sub {roc_auc_score(g.stage >= 2, 1 - g.d1):.3f}  cir {roc_auc_score(g.stage >= 4, g.d4):.3f}")

**Step 24.** CELL 11: in distribution against leave one scanner out, with matched seeds (Sec. S6).

In [ ]:
# CELL 11 (CPU, ~3 min, second notebook): C2 like-for-like, in-distribution vs LOSO, matched seeds, same metric
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
CKPT = "/content/drive/MyDrive/CARE2026_ckpts"; RES = f"{CKPT}/paper_results"
CENTERS, B = ["A", "B1", "B2"], 2000
RELIABLE = [("A", "sub"), ("A", "cir"), ("B1", "sub"), ("B1", "cir"), ("B2", "cir")]

# ---- in-distribution: same 5 folds as CELL J; RF exactly as CELL K ----
ID3 = pd.read_csv(f"{RES}/id_cv_deep_preds_3seeds.csv")
m = ID3[ID3.seed == 42][["filename", "vendor", "stage", "fold"]].reset_index(drop=True)      # CELL K's row order
F7 = pd.read_csv(f"{CKPT}/lifs_stad_fixed.csv"); cols = [c for c in F7.columns if c not in ("filename", "vendor", "stage")]
X = F7.set_index("filename").loc[m.filename, cols].values.astype(float)
rf = lambda: make_pipeline(SimpleImputer(strategy="median"), FunctionTransformer(np.nan_to_num),
                           RandomForestClassifier(n_estimators=600, min_samples_leaf=2, max_features="sqrt",
                                                  class_weight="balanced", n_jobs=-1, random_state=42))
for t, thr in [("sub", 2), ("cir", 4)]:
    yy = (m.stage.values >= thr).astype(int); oof = np.zeros(len(m))
    for k in sorted(m.fold.unique()):
        te = m.fold.values == k
        oof[te] = rf().fit(X[~te], yy[~te]).predict_proba(X[te])[:, 1]
    m[f"ID_rf_{t}"] = oof
d3 = ID3.groupby("filename")[["d1", "d4"]].mean(); d42 = ID3[ID3.seed == 42].set_index("filename")[["d1", "d4"]]
m["ID_deep3_sub"], m["ID_deep3_cir"] = 1 - d3.loc[m.filename, "d1"].values, d3.loc[m.filename, "d4"].values
m["ID_deep42_sub"], m["ID_deep42_cir"] = 1 - d42.loc[m.filename, "d1"].values, d42.loc[m.filename, "d4"].values
for t in ["sub", "cir"]:
    m[f"ID_fus3_{t}"] = 0.5 * m[f"ID_deep3_{t}"] + 0.5 * m[f"ID_rf_{t}"]
K = pd.read_csv(f"{RES}/K_in_distribution.csv"); K = K[(K.feat == "fixed7") & (K.scope == "pooled")].set_index("task")
for t, thr in [("sub", 2), ("cir", 4)]:
    yy = (m.stage.values >= thr).astype(int)
    print(f"REPRODUCTION CHECK {t}: deep seed 42 {roc_auc_score(yy, m[f'ID_deep42_{t}']):.4f} (CELL K {K.loc[t, 'deep']:.4f}) | "
          f"RF {roc_auc_score(yy, m[f'ID_rf_{t}']):.4f} (CELL K {K.loc[t, 'rf']:.4f})")

# ---- LOSO: the same models as CELL V ----
L = pd.read_csv(f"{CKPT}/lifs_loco_deep/loco_deep_preds.csv").groupby("filename")[["d1", "d4"]].mean()
BC = pd.read_csv(f"{RES}/B_loco_cases.csv"); BC = BC[(BC.feat == "fixed7") & (BC.variant == "raw")].set_index("filename")
m["LOSO_deep_sub"], m["LOSO_deep_cir"] = 1 - L.loc[m.filename, "d1"].values, L.loc[m.filename, "d4"].values
m["LOSO_rf_sub"], m["LOSO_rf_cir"] = BC.loc[m.filename, "rf_sub"].values, BC.loc[m.filename, "rf_cir"].values
for t in ["sub", "cir"]:
    m[f"LOSO_fus_{t}"] = 0.5 * m[f"LOSO_deep_{t}"] + 0.5 * m[f"LOSO_rf_{t}"]
M = m.sort_values("filename").reset_index(drop=True)        # CELL V's case order, so the bootstrap draws are the same

NAMES = {"ID_deep3": "in-dist deep (3 seeds)", "ID_deep42": "in-dist deep (seed 42, as submitted)",
         "ID_rf": "in-dist radiomic RF", "ID_fus3": "in-dist fusion (deep 3 seeds)",
         "LOSO_deep": "LOSO deep (3 seeds)", "LOSO_rf": "LOSO radiomic RF", "LOSO_fus": "LOSO fusion"}
ARR = {}
for h in CENTERS:
    g = M[M.vendor.astype(str) == h]
    ARR[h] = {"st": g.stage.values.astype(int), **{f"{k}_{t}": g[f"{k}_{t}"].values for k in NAMES for t in ("sub", "cir")}}
def cidx(k, idx):
    c = n = 0
    for h in CENTERS:
        st = ARR[h]["st"][idx[h]]; x = (ARR[h][f"{k}_sub"] + ARR[h][f"{k}_cir"])[idx[h]]
        comp = st[:, None] > st[None, :]; dx = (x[:, None] - x[None, :])[comp]
        c += (dx > 0).sum() + 0.5 * (dx == 0).sum(); n += comp.sum()
    return c / n
def auc5(k, idx):
    v = []
    for h, t in RELIABLE:
        ii = idx[h]; yy = (ARR[h]["st"][ii] >= (2 if t == "sub" else 4)).astype(int)
        if yy.min() == yy.max(): return np.nan
        v.append(roc_auc_score(yy, ARR[h][f"{k}_{t}"][ii]))
    return np.mean(v)
def pooled_auc(k, t, idx):
    yy = np.concatenate([ARR[h]["st"][idx[h]] >= (2 if t == "sub" else 4) for h in CENTERS]).astype(int)
    return roc_auc_score(yy, np.concatenate([ARR[h][f"{k}_{t}"][idx[h]] for h in CENTERS]))

rs = np.random.default_rng(0)
W = [{h: rs.integers(0, len(ARR[h]["st"]), len(ARR[h]["st"])) for h in CENTERS} for _ in range(B)]
full = {h: np.arange(len(ARR[h]["st"])) for h in CENTERS}
def summarize(dist, est):
    dist = dist[np.isfinite(dist)]; lo, hi = np.percentile(dist, [2.5, 97.5])
    return est, f"{lo:+.3f},{hi:+.3f}", max(1 / len(dist), min(1.0, 2 * min((dist <= 0).mean(), (dist >= 0).mean())))
COMPS = [("RF - deep, in-distribution (deep 3 seeds)", "ID_rf", "ID_deep3", None, None),
         ("RF - deep, in-distribution (deep seed 42, as submitted)", "ID_rf", "ID_deep42", None, None),
         ("RF - deep, LOSO", "LOSO_rf", "LOSO_deep", None, None),
         ("PROTOCOL EFFECT on RF - deep (LOSO minus in-dist, 3 seeds)", "LOSO_rf", "LOSO_deep", "ID_rf", "ID_deep3"),
         ("fusion - deep, in-distribution (3 seeds)", "ID_fus3", "ID_deep3", None, None),
         ("fusion - deep, LOSO", "LOSO_fus", "LOSO_deep", None, None),
         ("PROTOCOL EFFECT on fusion - deep", "LOSO_fus", "LOSO_deep", "ID_fus3", "ID_deep3"),
         ("deep: LOSO minus in-distribution (3 seeds)", "LOSO_deep", "ID_deep3", None, None),
         ("RF: LOSO minus in-distribution", "LOSO_rf", "ID_rf", None, None)]
vals, comps = [], []
for mname, fn in [("C-index (primary)", cidx), ("mean AUC over 5 reliable center-task pairs", auc5)]:
    pt = {k: fn(k, full) for k in NAMES}; bs = {k: np.array([fn(k, w) for w in W]) for k in NAMES}
    for k in NAMES:
        lo, hi = np.nanpercentile(bs[k], [2.5, 97.5]); vals.append(dict(metric=mname, model=NAMES[k], value=pt[k], ci=f"{lo:.3f}-{hi:.3f}"))
    for lab, a, b, c, d in COMPS:
        est = pt[a] - pt[b] - (0 if c is None else pt[c] - pt[d])
        dist = bs[a] - bs[b] - (0 if c is None else bs[c] - bs[d])
        e_, ci_, p_ = summarize(dist, est); comps.append(dict(metric=mname, comparison=lab, estimate=e_, ci=ci_, p=p_))
for t in ["sub", "cir"]:
    pt = {k: pooled_auc(k, t, full) for k in ["ID_rf", "ID_deep3", "ID_deep42"]}
    bs = {k: np.array([pooled_auc(k, t, w) for w in W]) for k in pt}
    for k in pt:
        lo, hi = np.percentile(bs[k], [2.5, 97.5]); vals.append(dict(metric=f"pooled AUC {t} (in-dist)", model=NAMES[k], value=pt[k], ci=f"{lo:.3f}-{hi:.3f}"))
    for lab, a, b in [(f"RF - deep (seed 42), pooled AUC {t}", "ID_rf", "ID_deep42"), (f"RF - deep (3 seeds), pooled AUC {t}", "ID_rf", "ID_deep3")]:
        e_, ci_, p_ = summarize(bs[a] - bs[b], pt[a] - pt[b]); comps.append(dict(metric=f"pooled AUC {t} (in-dist)", comparison=lab, estimate=e_, ci=ci_, p=p_))
VAL, CMP = pd.DataFrame(vals), pd.DataFrame(comps)
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 70)
print("\n=== values ===\n" + VAL.round(3).to_string(index=False))
print("\n=== comparisons (paired bootstrap, resampling patients within center, 2000 draws) ===\n" + CMP.round(4).to_string(index=False))
VAL.to_csv(f"{RES}/C2_values.csv", index=False); CMP.to_csv(f"{RES}/C2_comparisons.csv", index=False)
print("\nsaved C2_values.csv, C2_comparisons.csv")

**Step 25.** CELL 10: comparisons without the B2 fold, C index per held out scanner, and scanner decodability against transfer (Secs. S7 to S9).

In [ ]:
# CELL 10 (CPU, ~10-15 min, second notebook): C3 re-analysis from saved predictions (no training)
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, os, warnings
import matplotlib
matplotlib.rcParams["pdf.fonttype"] = 42; matplotlib.rcParams["ps.fonttype"] = 42
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
CKPT = "/content/drive/MyDrive/CARE2026_ckpts"; RES = f"{CKPT}/paper_results"; DG = f"{CKPT}/lifs_dg"
CENTERS, B = ["A", "B1", "B2"], 2000
INTERV = ["multiphase", "intensity_aug", "adversarial"]

# ---------- 1. every model's held-out risk score, built exactly as in CELL V ----------
dp = pd.read_csv(f"{CKPT}/lifs_loco_deep/loco_deep_preds.csv")
D = dp.groupby(["filename", "held", "stage"], as_index=False)[["d1", "d4"]].mean()
D["deep"] = (1 - D.d1) + D.d4
BC = pd.read_csv(f"{RES}/B_loco_cases.csv")
for feat in ["fixed7", "ged4"]:
    for var in ["raw", "center-z", "center-rank"]:
        g = BC[(BC.feat == feat) & (BC.variant == var)]
        if len(g) == 0: continue
        D = D.merge(pd.DataFrame({"filename": g.filename.values, f"rf_{feat}_{var}": (g.rf_sub + g.rf_cir).values}),
                    on="filename", how="left")
        D[f"fusion_{feat}_{var}"] = 0.5 * D.deep + 0.5 * D[f"rf_{feat}_{var}"]
e = np.load(f"{RES}/fm_dinov2_embeddings.npz", allow_pickle=True)
Xe, Ve, se, fe = e["emb"], e["vendor"].astype(str), e["stage"].astype(int), e["filename"].astype(str)
lr = lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000))
s = np.zeros(len(fe))
for h in CENTERS:
    tr = Ve != h
    for thr in (2, 4): s[~tr] += lr().fit(Xe[tr], se[tr] >= thr).predict_proba(Xe[~tr])[:, 1]
D = D.merge(pd.DataFrame({"filename": fe, "dinov2": s}), on="filename", how="left")
D = D.merge(pd.read_csv(f"{RES}/U_combat_cases.csv")[["filename", "rf_combat", "logreg_combat"]], on="filename", how="left")
MV = [c for c in D.columns if c not in ("filename", "held", "stage", "d1", "d4")]          # CELL V's model list
for mth in INTERV:
    g = pd.read_csv(f"{DG}/dg_preds_{mth}.csv").groupby("filename", as_index=False)[["d1", "d4"]].mean()
    D = D.merge(pd.DataFrame({"filename": g.filename, mth: (1 - g.d1) + g.d4}), on="filename", how="left")
MA = MV + INTERV
assert not D[MA].isna().any().any(), "missing predictions"
print(f"{len(D)} cases | {len(MV)} models from CELL V + {len(INTERV)} interventions")

A = {h: (D[D.held == h].stage.values, {m: D[D.held == h][m].values for m in MA}) for h in CENTERS}
def pooled(idx, centers, ms):
    c = {m: 0.0 for m in ms}; n = 0
    for h in centers:
        st = A[h][0][idx[h]]; comp = st[:, None] > st[None, :]; n += comp.sum()
        for m in ms:
            x = A[h][1][m][idx[h]]; dx = (x[:, None] - x[None, :])[comp]
            c[m] += (dx > 0).sum() + 0.5 * (dx == 0).sum()
    return np.array([c[m] / n for m in ms])
full = {h: np.arange(len(A[h][0])) for h in CENTERS}
def draws(seed, centers):
    r = np.random.default_rng(seed)
    return [{h: r.integers(0, len(A[h][0]), len(A[h][0])) for h in centers} for _ in range(B)]
def pval(d): return max(1 / B, min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))
def ci(v): lo, hi = np.percentile(v, [2.5, 97.5]); return lo, hi

# ---------- 2. primary endpoint, every model vs deep (same bootstrap draws as CELL V) ----------
point = pooled(full, CENTERS, MV); boots = np.array([pooled(w, CENTERS, MV) for w in draws(0, CENTERS)])
j = {m: i for i, m in enumerate(MV)}; FAM1 = ("fusion_fixed7_raw", "rf_fixed7_raw", "dinov2"); rows = []
for m in MV:
    lo, hi = ci(boots[:, j[m]]); d = boots[:, j[m]] - boots[:, j["deep"]]; dlo, dhi = ci(d)
    rows.append(dict(model=m, C=point[j[m]], C_ci=f"{lo:.3f}-{hi:.3f}", above_chance=lo > 0.5,
                     delta=np.nan if m == "deep" else point[j[m]] - point[j["deep"]],
                     dlo=np.nan if m == "deep" else dlo, dhi=np.nan if m == "deep" else dhi,
                     p_unadj=np.nan if m == "deep" else pval(d),
                     family="1 (pre-declared, Holm in CELL V)" if m in FAM1 else "descriptive"))
T = pd.DataFrame(rows).sort_values("C", ascending=False).reset_index(drop=True)
old = pd.read_csv(f"{RES}/V_concordance.csv").set_index("model"); chk = T.set_index("model").join(old[["C_index", "ci"]])
print("REPRODUCTION CHECK vs CELL V: max |C diff| =", f"{(chk.C - chk.C_index).abs().max():.1e}",
      "| all CIs identical:", bool((chk.C_ci == chk.ci).all()))
print("\n=== (2) LOSO, every model vs the deep stream (pooled within-center C-index, 95% CI) ===")
print(T.round(3).to_string(index=False))
alt = T[T.model != "deep"]; inc0 = alt[(alt.dlo <= 0) & (alt.dhi >= 0)]
print("\nsignificantly WORSE than deep:", list(alt[alt.dhi < 0].model))
print("significantly BETTER than deep:", list(alt[alt.dlo > 0].model))
print(f"CI includes 0: {len(inc0)} models; largest upper bound = {inc0.dhi.max():+.3f} ({inc0.loc[inc0.dhi.idxmax(), 'model']})")

# ---------- 3. per held-out center: C-index with 95% CI, is it above chance? ----------
rows = []
for k, h in enumerate(CENTERS):
    W = draws(100 + k, [h]); pt = pooled(full, [h], MA); bt = np.array([pooled(w, [h], MA) for w in W])
    for i, m in enumerate(MA):
        lo, hi = ci(bt[:, i]); rows.append(dict(center=h, model=m, C=pt[i], lo=lo, hi=hi, above_chance=lo > 0.5))
PC = pd.DataFrame(rows)
PC["cell"] = PC.apply(lambda r: f"{r.C:.3f} [{r.lo:.2f},{r.hi:.2f}]{'*' if r.above_chance else ''}", axis=1)
print("\n=== (3) per held-out center (* = 95% CI above 0.5) ===")
print(PC.pivot(index="model", columns="center", values="cell").loc[MA].to_string())

# ---------- 4. post hoc sensitivity: A and B1 folds only ----------
W2 = draws(2, ["A", "B1"]); pt2 = pooled(full, ["A", "B1"], MA); bt2 = np.array([pooled(w, ["A", "B1"], MA) for w in W2])
jj = {m: i for i, m in enumerate(MA)}; rows = []
for m in MA:
    d = bt2[:, jj[m]] - bt2[:, jj["deep"]]; dlo, dhi = ci(d)
    rows.append(dict(model=m, C_A_B1=pt2[jj[m]], delta=pt2[jj[m]] - pt2[jj["deep"]], dlo=dlo, dhi=dhi,
                     p_unadj=np.nan if m == "deep" else pval(d)))
AB = pd.DataFrame(rows).sort_values("C_A_B1", ascending=False)
print("\n=== (4) post hoc: A and B1 folds only (B2 excluded) ===")
print(AB.round(3).to_string(index=False))

# ---------- 5. does scanner information predict transfer? ----------
def vendor_dec(emb, V):
    cl = np.unique(V)
    P = cross_val_predict(RandomForestClassifier(n_estimators=600, class_weight="balanced", n_jobs=-1, random_state=42),
                          emb, V, cv=StratifiedKFold(5, shuffle=True, random_state=42), method="predict_proba")
    return roc_auc_score(V, P, multi_class="ovr", labels=cl)
Cm = dict(zip(MA, pooled(full, CENTERS, MA))); pts = []
eb = np.load(f"{RES}/deep_embeddings.npz", allow_pickle=True)
pts.append(("deep GED4 (baseline)", "trained deep", vendor_dec(eb["emb"], eb["vendor"].astype(str)), Cm["deep"], "recomputed"))
for mth, lab in [("multiphase", "multiphase"), ("intensity_aug", "intensity aug."), ("adversarial", "adversarial")]:
    em = np.load(f"{DG}/emb_{mth}_B2_seed42.npz", allow_pickle=True)
    pts.append((lab, "trained deep", vendor_dec(em["emb"], em["vendor"].astype(str)), Cm[mth], "recomputed"))
pts.append(("frozen DINOv2", "frozen foundation model", vendor_dec(Xe, Ve), Cm["dinov2"], "recomputed"))
for _, r in pd.read_csv(f"{RES}/D_vendor_predictability.csv").iterrows():
    pts.append((f"RF {r.source} {r.variant}", "radiomics", r.vendor_auc, Cm[f"rf_{r.source}_{r.variant}"], "D_vendor_predictability.csv"))
pts.append(("RF ComBat", "radiomics", 0.999, Cm["rf_combat"], "printed by CELL U, Oct 16"))
pts.append(("logreg ComBat", "radiomics", 0.999, Cm["logreg_combat"], "printed by CELL U, Oct 16"))
DT = pd.DataFrame(pts, columns=["representation", "family", "scanner_auc", "loso_C", "scanner_auc_source"])
print("\n=== (5) scanner information vs transfer ===")
print(DT.round(3).to_string(index=False))
rho, p = spearmanr(DT.scanner_auc, DT.loso_C); dd = DT[DT.family == "trained deep"]
print(f"Spearman, all {len(DT)} representations: rho {rho:+.2f} (p {p:.3f}) | "
      f"trained deep only (n={len(dd)}): rho {spearmanr(dd.scanner_auc, dd.loso_C)[0]:+.2f}")

fig, ax = plt.subplots(figsize=(3.4, 2.8)); mk = {"trained deep": "o", "frozen foundation model": "s", "radiomics": "^"}
for fam, g in DT.groupby("family"):
    ax.scatter(g.scanner_auc, g.loso_C, marker=mk[fam], s=22, label=fam)
for _, r in DT[DT.family != "radiomics"].iterrows():
    ax.annotate(r.representation, (r.scanner_auc, r.loso_C), fontsize=5, xytext=(3, 2), textcoords="offset points")
ax.axhline(0.5, ls="--", lw=0.6, c="grey")
ax.set_xlabel("Scanner AUC (lower = less scanner information)", fontsize=7); ax.set_ylabel("LOSO C-index", fontsize=7)
ax.tick_params(labelsize=6); ax.legend(fontsize=5, frameon=False, loc="lower left"); fig.tight_layout()
fig.savefig(f"{RES}/fig_decodability_vs_transfer.pdf"); fig.savefig(f"{RES}/fig_decodability_vs_transfer.png", dpi=200); plt.show()

T.to_csv(f"{RES}/C3_all_vs_deep.csv", index=False); PC.to_csv(f"{RES}/C3_per_center.csv", index=False)
AB.to_csv(f"{RES}/C3_A_B1_only.csv", index=False); DT.to_csv(f"{RES}/C3_decodability_vs_transfer.csv", index=False)
print("\nsaved: C3_all_vs_deep.csv, C3_per_center.csv, C3_A_B1_only.csv, C3_decodability_vs_transfer.csv, fig_decodability_vs_transfer.pdf")

**Step 26.** CELL 15: corruption simulation (Sec. S10).

In [ ]:
# CELL 15 (first notebook, after CELL 14, ~20-30 min unattended): C5 controlled corruption experiment
# Post hoc, reviewer-requested. Design fixed before running; report whatever comes out:
#   radiomic forest (as CELL K, but 300 trees for speed) on the clean 7-phase features (lifs_stad_fixed.csv)
#   corrupt a share p of ONE scanner's exams; p in {10, 25, 50}%; target scanner A, B1, B2; 3 random draws
#   corruption: zero 3 phases | zero all 7 phases (the empty-crop artifact) | shift +1 SD | shift +2 SD
#   which exams: random within the scanner | stage-linked (chance proportional to stage, S4 four times S1)
#   compare: pooled 5-fold CV (CELL J folds) vs leaving the target scanner out; primary endpoint (within-center C-index)
import numpy as np, pandas as pd, time, warnings
import matplotlib
matplotlib.rcParams["pdf.fonttype"] = 42; matplotlib.rcParams["ps.fonttype"] = 42
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
warnings.filterwarnings("ignore")
CKPT = "/content/drive/MyDrive/CARE2026_ckpts"; RES = f"{CKPT}/paper_results"
F = pd.read_csv(f"{CKPT}/lifs_stad_fixed.csv").merge(pd.read_csv(f"{RES}/id_cv_deep_preds.csv")[["filename", "fold"]], on="filename")
assert len(F) == 460
cols = [c for c in F.columns if c not in ("filename", "vendor", "stage", "fold")]
PHASES = ["T1", "T2", "DWI_800", "GED1", "GED2", "GED3", "GED4"]
PCOL = {ph: [i for i, c in enumerate(cols) if c.startswith(ph + "_")] for ph in PHASES}
print(f"{len(cols)} features | per phase: { {p: len(v) for p, v in PCOL.items()} }")
X0 = F[cols].to_numpy(dtype=float); V = F.vendor.astype(str).to_numpy(dtype=str)
st = F.stage.to_numpy(dtype=int); fold = F.fold.to_numpy(dtype=int); SD = np.nanstd(X0, 0)
rf = lambda: make_pipeline(SimpleImputer(strategy="median"), FunctionTransformer(np.nan_to_num),
                           RandomForestClassifier(n_estimators=300, min_samples_leaf=2, max_features="sqrt",
                                                  class_weight="balanced", n_jobs=-1, random_state=42))
def risk(Xtr, s_tr, Xte):
    return sum(rf().fit(Xtr, s_tr >= t).predict_proba(Xte)[:, 1] for t in (2, 4))
def cidx(score, stage, groups):
    c = n = 0
    for g in np.unique(groups):
        m = groups == g; s, x = stage[m], score[m]
        comp = s[:, None] > s[None, :]; dx = (x[:, None] - x[None, :])[comp]
        c += (dx > 0).sum() + 0.5 * (dx == 0).sum(); n += comp.sum()
    return c / n
def run(X, target):
    pooled = np.zeros(len(X))
    for k in np.unique(fold):
        te = fold == k; pooled[te] = risk(X[~te], st[~te], X[te])
    te = V == target; loso = risk(X[~te], st[~te], X[te])
    return dict(C_pooled_all=cidx(pooled, st, V), C_pooled_target=cidx(pooled[te], st[te], V[te]),
                C_loso_target=cidx(loso, st[te], V[te]))
KINDS = [("zero 3 phases", "zero", 3), ("zero all 7 phases", "zero", 7), ("shift +1 SD", "shift", 1.0), ("shift +2 SD", "shift", 2.0)]
rows, t0 = [], time.time()
for target in ["A", "B1", "B2"]:
    it = np.where(V == target)[0]
    rows.append(dict(target=target, kind="none", selection="none", prevalence=0.0, draw=0, **run(X0, target)))
    for ki, (kname, ktype, kval) in enumerate(KINDS):
        for selection in ["random", "stage-linked"]:
            for p in [0.10, 0.25, 0.50]:
                for draw in range(3):
                    r = np.random.default_rng([draw, int(100 * p), ki, ["A", "B1", "B2"].index(target), int(selection == "random")])
                    w = None if selection == "random" else st[it] / st[it].sum()
                    sel = r.choice(it, int(round(p * len(it))), replace=False, p=w)
                    X = X0.copy()
                    if ktype == "zero":
                        for ph in r.choice(PHASES, kval, replace=False):
                            X[np.ix_(sel, PCOL[ph])] = 0.0
                    else:
                        X[sel] = X[sel] + kval * SD
                    rows.append(dict(target=target, kind=kname, selection=selection, prevalence=p, draw=draw, **run(X, target)))
    print(f"  target {target} done ({time.time()-t0:.0f}s)")
R = pd.DataFrame(rows)
B0 = R[R.kind == "none"].set_index("target")
for c in ["C_pooled_all", "C_pooled_target", "C_loso_target"]:
    R[f"chg_{c}"] = R[c] - R.target.map(B0[c])
R.to_csv(f"{RES}/C5_corruption_runs.csv", index=False)
print("\nuncorrupted reference (300-tree forest):\n" + B0[["C_pooled_all", "C_pooled_target", "C_loso_target"]].round(3).to_string())
S = (R[R.kind != "none"].groupby(["kind", "selection", "prevalence"])[["chg_C_pooled_all", "chg_C_pooled_target", "chg_C_loso_target"]]
     .agg(["mean", "std"]).round(3))
print("\nchange in C-index vs uncorrupted, mean and SD over 3 scanners x 3 draws:\n" + S.to_string())
S.to_csv(f"{RES}/C5_corruption_summary.csv")

fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.6), sharey=True)
for ax, selection in zip(axes, ["random", "stage-linked"]):
    for kname, _, _ in KINDS:
        g = R[(R.kind == kname) & (R.selection == selection)].groupby("prevalence")[["chg_C_pooled_all", "chg_C_loso_target"]].mean()
        l, = ax.plot(100 * g.index, g.chg_C_loso_target, marker="o", ms=3, lw=1, label=kname)
        ax.plot(100 * g.index, g.chg_C_pooled_all, ls="--", lw=1, color=l.get_color())
    ax.axhline(0, color="grey", lw=0.5); ax.set_title(f"{selection} exams corrupted", fontsize=7)
    ax.set_xlabel("% of the target scanner's exams corrupted", fontsize=7); ax.tick_params(labelsize=6)
axes[0].set_ylabel("change in C-index\n(solid: target scanner held out; dashed: pooled CV)", fontsize=6)
axes[1].legend(fontsize=5, frameon=False); fig.tight_layout()
fig.savefig(f"{RES}/fig_corruption.pdf"); fig.savefig(f"{RES}/fig_corruption.png", dpi=200); plt.show()
print("saved C5_corruption_runs.csv, C5_corruption_summary.csv, fig_corruption.pdf")